# Module 2: Defect extraction, validation and banking

Takes the normal and anomaly image pairs from Module 1, recovers the defect mask from each pair,
validates the result, and writes a defect bank for Module 3.

The mask is recovered from the pair rather than prescribed in advance, so the label always matches
the image.

## Requirements

Kaggle notebook, accelerator set to GPU T4 x2, internet off.

Attach two mounts:

1. `Qwen/Qwen2.5-VL-7B-Instruct`, as a Model.
2. Your Module 1 output, as a Dataset, laid out as `<category>/<id>_regular.png` and
   `<category>/<id>_anomaly.png`.

## Steps

1. Open section 1 and set the mount paths and the output directory.
2. Run all cells in order. Expect a few seconds per pair for mask recovery and a few seconds per
   crop for validation.
3. Check the overlays in section 5. Every candidate region is drawn on the frame it came from.
4. Check the verdict overlay in section 8, which colours each crop by whether it was banked.
5. Read the funnel in section 11. A category reaching zero cannot be used by Module 3.

## Output

```
defect_bank/
  <category>/<key>.png            the defect crop, with its surrounding substrate
  <category>/<key>_alpha.png      the defect mask for that crop
  manifest.csv                    geometry, gate results, model verdict, timing

Defect Masks and Anomalies/
  <category>/<id>_anomaly.png     the generated anomaly frame
  <category>/<id>_mask.png        the mask recovered from it
```

Both files in the bank are required by Module 3. The crop keeps its background because the
harmoniser measures it, and the mask becomes the ground truth.

## 1. Config

Set the mount paths and the output directory here.

### DiffMask flags

The same four flags run on every category. They were fitted on 12 pairs and are not the library
defaults, which fail on this data.

| flag | value | effect |
|---|---|---|
| `--w-sharp` | `0` | disables the sharpness cue, which fires on re-render differences rather than defects |
| `--shrink` | `0.4` | controls post-detection erosion; higher values fragment thin regions |
| `--norm-frac` | `0.10` | width of the local noise window |
| `--auto-low` | `2.5` | enables the low-frequency cue, needed for low-contrast foreign objects |

`--sat-guard` stays at `0.99`. It drops pixels that are clipped in every channel in either frame,
which removes false positives where one frame blows out to white and the other does not.

### Known limitation

DiffMask's score is sized for compact regions. A defect thinner than the score blur, or one that
differs from its background in only one colour channel, may not be recovered at all. Such a pair
produces an empty mask, is reported as `EMPTY` in section 4, and contributes nothing to the bank.

If you need thin defect recovery, raise `--combine-p` and lower `--smooth`, then re-check every
category: the change affects all of them.

In [ ]:
import os, sys, json, glob, math, time, shutil, zipfile, argparse, textwrap, random
import numpy as np
import cv2
from pathlib import Path

SEED = 0
random.seed(SEED); np.random.seed(SEED)

# ---------------------------------------------------------------------------- mounts
# Both are searched if the literal path is absent, because Kaggle nests Dataset and Model
# mounts differently (models land under /kaggle/input/models/<owner>/<name>/<fw>/<var>/<ver>).
# Dataset ref OR literal path. A kaggle URL and an "owner/slug" both work: mount_candidates()
# in section 2 expands them, because Kaggle mounts the same dataset at different paths on
# different kernels (/kaggle/input/<slug> vs /kaggle/input/datasets/<owner>/<slug>).
PAIRS_ID = "abhaykdas/local-testing"              # <cat>/<id>_regular.png + <id>_anomaly.png
MODEL_ID = "/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/7b-instruct/2"
# TWO output folders, and nothing else.
#   BANK_DIR   accepted cropped defect patches only  -> Stages 4-5 read this
#   PAIRS_DIR  category-wise (recovered mask, generated anomaly) pairs, one per donor
OUT_ROOT  = "/kaggle/working"
BANK_DIR  = os.path.join(OUT_ROOT, "defect_bank")
PAIRS_DIR = os.path.join(OUT_ROOT, "Defect Masks and Anomalies")
WORK_DIR  = os.path.join(OUT_ROOT, "_work")       # diffmask scratch; not a deliverable

# diffmask's 7 intermediates per pair. Off by default -- they are a debugging aid, not output.
WRITE_DEBUG = False

CATS = ["can", "fabric", "fruit_jelly", "rice",
        "sheet_metal", "vial", "wallplugs", "walnuts"]

# Supplementary 3.A: the configuration was fixed on four development categories and applied
# unchanged to four held-out ones. Local Testing contains BOTH, so every table below marks
# which is which -- a mean over all eight is not a held-out result and must not be read as one.
DEV_CATS     = {"rice", "walnuts", "wallplugs", "fruit_jelly"}
HELDOUT_CATS = {"can", "fabric", "sheet_metal", "vial"}

# Coarse prior on what a defect looks like per category. Fed to VLM-2 as context so it judges
# "is this a plausible <kind> defect on <cat>" rather than "is this interesting".
DEFECT_KIND = {"rice": "foreign_object",   "walnuts": "surface",
               "wallplugs": "foreign_object", "fruit_jelly": "foreign_object",
               "can": "surface",           "fabric": "surface",
               "sheet_metal": "surface",   "vial": "foreign_object"}

# ---------------------------------------------------------------------- diffmask flags
# Frozen config from run_diffmask_batch.py, fixed on four development categories and applied
# unchanged to all eight. There is deliberately NO per-category override table: supplementary
# 3.A claims a fixed, category-agnostic configuration, and a rescue config for one category
# would contradict it. Do not edit without re-running every pair.
FLAGS = ["--w-sharp", "0", "--shrink", "0.4", "--norm-frac", "0.10", "--auto-low", "2.5"]

# ------------------------------------------------------------------------- bank gates
WORK_SIZE       = 1024   # paper Table 3, synthesis resolution
CTX_EXPAND      = 1.8    # crop side = CTX_EXPAND x defect bbox long side (substrate retained)
MIN_REGION_PX   = 500    # recovered components below this are diffmask speckle
ALPHA_SOFTEN    = 1.0    # anti-alias sigma on the hard recovered mask edge
DEFECT_T        = 6.0    # Lab distance above which a crop pixel counts as defect, not substrate
MIN_ENTRY_CFRAC = 0.10   # fraction of masked pixels that must clear DEFECT_T

# ------------------------------------------------------------------------------- VLM
LOAD_4BIT      = False           # False -> fp16 split across both T4s, no internet needed
# Visual-token cap. Raised from 1280 because section 7 now sends THREE panels side by side:
# at the old budget each panel got ~578px and the defect inside it ~180px, which is smaller
# than it was under the old single-panel prompt. Context is useless if it costs the acuity to
# see what you are judging. 2048 patches -> ~730px per panel.
MAX_PIXELS     = 2048 * 28 * 28
MIN_PIXELS     = 256 * 28 * 28
MAX_NEW_TOKENS = 220             # the JSON verdict is short; this is headroom
VLM_MIN_CONF   = 0.5             # accept threshold on VLM-2's own confidence

# Stage 1 introduces exactly ONE defect per generated image. So when DiffMask returns several
# components from one donor, at most one of them is that defect and the rest are extraction
# artifacts -- typically a registration seam, which shows up as a thin elongated strip.
# With this on, VLM-2 is told the region count and its rank, and at most one crop per donor is
# banked: the accepted one it was most confident about. Turn it off only if the generator is
# ever asked for scattered or multi-site defects, where several components are one defect.
ONE_DEFECT_PER_IMAGE = True

# VLM-2 sees a slightly wider crop than the one that gets banked, so some clean substrate stays
# in frame to compare against. This only affects what the model is SHOWN; the banked crop is
# still CTX_EXPAND. Kept modest on purpose: every extra unit of context shrinks the defect
# inside a fixed token budget, and 3.2 cost more acuity than the context was worth.
#   defect size in the model's view ~= (panel px) / VLM_CTX
VLM_CTX = 2.2

# Gate on VLM-2's verdict alone, not on its self-reported confidence. A 7B VLM's confidence is
# close to uncalibrated -- it clusters at 0.8-0.9 regardless of whether it is right -- so
# thresholding it mostly removes correct answers. The number is still recorded, and still used
# to break ties between two regions of the SAME donor (a relative comparison, which is the one
# thing it is fit for). Set True only after checking the confidence histogram is not flat.
USE_VLM_CONFIDENCE = False

for _d in (BANK_DIR, PAIRS_DIR, WORK_DIR):
    os.makedirs(_d, exist_ok=True)
print(f"cats={len(CATS)} ctx={CTX_EXPAND} min_region={MIN_REGION_PX}px "
      f"dE={DEFECT_T} cfrac>={MIN_ENTRY_CFRAC}")
print("flags:", " ".join(FLAGS), " (identical for every category)")

## 2. Pair discovery

Finds the image pairs and reports what it found.

Expected layout, one directory per category:

```
<root>/<category>/<id>_regular.png     the normal image
<root>/<category>/<id>_anomaly.png     the generated anomaly
```

A pair is used only when both files exist. An anomaly with no matching normal is listed separately
so a failed generation is visible rather than silently absent.

The two images do not need to be aligned, the same size, or the same exposure. Registration is
handled in section 3. Do not resize them to match.

If the configured path is not found, the notebook searches for the directory holding the most
complete pairs and prints the one it chose. Check that line.

In [ ]:
def mount_candidates(hint):
    """Turn a dataset reference into the paths Kaggle might actually have mounted it at.

    Accepts a full URL, an "owner/slug" ref, or a literal path. Kaggle is not consistent about
    the mount point -- the same dataset appears as /kaggle/input/<slug> on some kernels and
    /kaggle/input/datasets/<owner>/<slug> on others -- so every plausible form is tried before
    falling back to a walk.
    """
    if not hint:
        return []
    h = hint.strip().rstrip("/\\")
    # A path that already exists wins outright. Checked BEFORE the "/" test, because a Windows
    # path (C:\...) is absolute without starting with "/" and would otherwise be mangled into
    # a bogus /kaggle/input/<drive> candidate.
    if os.path.isdir(h):
        return [h]
    if h.startswith("http"):                       # .../datasets/<owner>/<slug>[/...]
        parts = [x for x in h.split("/") if x]
        if "datasets" in parts:
            k = parts.index("datasets")
            parts = parts[k + 1:k + 3]
            h = "/".join(parts)
    if os.path.isabs(h):
        return [h]
    bits = [b for b in h.split("/") if b]
    slug = bits[-1]
    owner = bits[-2] if len(bits) >= 2 else None
    cands = [f"/kaggle/input/{slug}"]
    if owner:
        cands += [f"/kaggle/input/{owner}/{slug}",
                  f"/kaggle/input/datasets/{owner}/{slug}"]
    return cands


def find_pairs_root(hint=PAIRS_ID, max_depth=8):
    """Use `hint` if it holds category folders, else walk /kaggle/input for the best candidate.

    "Best" = the directory with the most immediate children that look like a category dir,
    i.e. contain at least one *_anomaly.png. Weight directories are pruned so the walk does
    not descend into model shards.
    """
    def score(root):
        """Number of COMPLETE pairs under `root`, not the number of folders holding anomalies.

        Counting folders picks the wrong directory whenever something else in the tree also
        holds *_anomaly.png -- a previous run's output, for instance, which has the anomaly
        copies but none of the _regular references and therefore yields zero usable pairs.
        """
        n = 0
        for d in sorted(os.listdir(root)):
            p = os.path.join(root, d)
            if not os.path.isdir(p):
                continue
            for a in glob.glob(os.path.join(p, "*_anomaly.png")):
                if os.path.exists(a.replace("_anomaly.png", "_regular.png")):
                    n += 1
        return n

    for cand in mount_candidates(hint):
        if os.path.isdir(cand) and score(cand):
            print(f"  pairs: {cand}")
            return cand
    if hint:
        print(f"  no complete pairs at any mount form of {hint!r} -- searching instead")

    best, best_n = None, 0
    base = "/kaggle/input" if os.path.isdir("/kaggle/input") else "."
    for root, dirs, _ in os.walk(base):
        if root.count(os.sep) - base.count(os.sep) > max_depth:
            dirs[:] = []; continue
        dirs[:] = [d for d in dirs if not d.startswith(".")
                   and not glob.glob(os.path.join(root, d, "*.safetensors"))]
        try:
            n = score(root)
        except OSError:
            continue
        if n > best_n:                    # best, not first: keep walking past a partial match
            best, best_n = root, n
    if best is None:
        raise FileNotFoundError(
            f"no <cat>/<id>_regular.png + <id>_anomaly.png pairs found under {base} -- attach "
            "the dataset, or point PAIRS_ID at it")
    print(f"  found {best_n} complete pairs under {best}")
    return best


PAIRS_ROOT = find_pairs_root()

PAIRS = []          # (cat, pid, ref_path, defect_path)
ORPHANS = []        # a prompt whose generation is missing -- reported, never silently dropped
for cat in sorted(os.listdir(PAIRS_ROOT)):
    cdir = os.path.join(PAIRS_ROOT, cat)
    if not os.path.isdir(cdir):
        continue
    for dfc in sorted(glob.glob(os.path.join(cdir, "*_anomaly.png"))):
        pid = os.path.basename(dfc)[: -len("_anomaly.png")]
        ref = os.path.join(cdir, f"{pid}_regular.png")
        (PAIRS if os.path.exists(ref) else ORPHANS).append((cat, pid, ref, dfc))

if not PAIRS:
    raise SystemExit(f"no complete pairs under {PAIRS_ROOT}")

_by_cat = {}
for cat, pid, _, _ in PAIRS:
    _by_cat.setdefault(cat, []).append(pid)
print(f"\n{len(PAIRS)} pairs across {len(_by_cat)} categories (root: {PAIRS_ROOT})")
for c in sorted(_by_cat):
    split = "dev" if c in DEV_CATS else ("held-out" if c in HELDOUT_CATS else "?")
    print(f"  {c:14s} {split:9s} {len(_by_cat[c]):3d}  {', '.join(_by_cat[c][:8])}"
          f"{' ...' if len(_by_cat[c]) > 8 else ''}")
if ORPHANS:
    print(f"\n{len(ORPHANS)} anomaly frames with no _regular reference:")
    for c, p, _, _ in ORPHANS[:12]:
        print(f"  {c}/{p}")

## 3. DiffMask

The next twelve cells define the mask recovery code. Run them in order; they define functions and
produce no output.

Recovery works in six steps: coarse scale and translation search on edge maps, ECC refinement with
a rotation-capable fallback, optional dense flow for residual drift, photometric matching with a
tolerance band, five difference cues fused into one score, and a local z-score followed by
hysteresis and component selection.

| Algorithm 1 step | section | function |
|---|---|---|
| 1. scale and translation search | 3.5 | `coarse_similarity` |
| 2. ECC refinement, feature fallback | 3.5 | `ecc_refine`, `feature_similarity`, `register` |
| 3. dense flow | 3.6 | `flow_refine` |
| 4. photometric match, tolerance band | 3.7 | `photometric_fit`, `band_residual` |
| 5. five residuals fused | 3.7 to 3.9 | `low_freq_*`, `sharpness_z`, `difference_score` |
| 6. local statistics | 3.8 | `local_zscore` |
| 7. hysteresis | 3.10 | `hysteresis` |
| 8. component ranking | 3.10 | `select_components` |
| 9. direction-matched completion | 3.10 | `complete_object` |

### 3.1 Module header and imports

In [ ]:
# ============================================================================
# diffmask (1).py lines 1-45 -- module docstring, imports, constants.
#
# The docstring is the six-step pipeline the paper's Algorithm 1 formalises. EPS is the shared
# divide-by-zero floor; every ratio in the file uses it rather than a local literal.
# ============================================================================
#!/usr/bin/env python3
"""
diffmask - find the extra object.

Takes a reference image and a defect image of the same scene and emits a binary
mask (white = object present only in the defect image) in the defect image's own
pixel frame.

The two inputs do not have to be aligned, the same size, or the same exposure.
Resizing them to a common size does not help and is not what the size mismatch
means: the subject sits at a different scale and offset inside each frame, so
matching the canvases still leaves the content tens of pixels apart.

The pipeline is:

  1. coarse scale + translation search on edge maps (multi-scale template match)
  2. ECC refinement to an affine or homography warp
  3. optional low-frequency dense flow to soak up non-rigid drift
  4. photometric-invariant difference: a tolerance band that forgives sub-pixel
     misregistration, over luma, chroma and gradient
  5. threshold against a local noise estimate, since a re-render is noisy in
     textured areas and silent on flat ones
  6. completion, which grows each region into the rest of the same object by
     matching the direction of its difference rather than the size

Usage:
    python diffmask.py ref.png defect.png -o mask.png
    python diffmask.py ref.png defect.png -o mask.png --overlay seen.png
    python diffmask.py ref.png defect.png -o mask.png --debug dbg/
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

import cv2
import numpy as np

EPS = np.float32(1e-6)

cv2.setUseOptimized(True)

### 3.2 Image read and write

In [ ]:
# ============================================================================
# diffmask (1).py lines 46-68 -- image read/write.
#
# Goes through np.fromfile/imdecode rather than cv2.imread because cv2's own path handling is
# byte-oriented and breaks on non-ascii Windows paths. Irrelevant on Kaggle, kept because
# diverging from the file under test to save two lines is a bad trade.
# ============================================================================

# --------------------------------------------------------------------------- io


def imread(path: str) -> np.ndarray:
    """Read via numpy so non-ascii Windows paths work."""
    buf = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if img is None:
        sys.exit(f"diffmask: cannot read image: {path}")
    return img


def imwrite(path: str, img: np.ndarray) -> None:
    p = Path(path)
    if p.parent and str(p.parent) not in ("", "."):
        p.parent.mkdir(parents=True, exist_ok=True)
    ok, buf = cv2.imencode(p.suffix or ".png", img)
    if not ok:
        sys.exit(f"diffmask: cannot encode: {path}")
    buf.tofile(str(p))

### 3.3 Small utilities

In [ ]:
# ============================================================================
# diffmask (1).py lines 69-152 -- greyscale, resize, edge maps, robust statistics.
#
# Two functions here carry real decisions:
#
# edge_pair vs edge_map. edge_map normalises each image by its OWN 99.5th percentile, which is
# what makes it tone invariant -- correct when the two maps are never compared. When they are
# subtracted it is wrong: the warped reference carries a hard step where its coverage runs out,
# that step sets its percentile, and every real edge in the reference comes out 2-3x weaker than
# the identical edge in the test. The whole silhouette then reads as a difference. edge_pair uses
# one normaliser measured only over the overlap.
#
# robust_scale's `floor`. Without a physically meaningful noise floor, a near-identical pair
# drives sigma to zero and every rounding error becomes an enormous z-score.
# ============================================================================
# ------------------------------------------------------------------ small utils


def to_gray32(bgr: np.ndarray) -> np.ndarray:
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255.0


def fit_long_side(shape, target: int) -> float:
    h, w = shape[:2]
    return min(1.0, target / float(max(h, w))) if target > 0 else 1.0


def resize_f(img: np.ndarray, f: float) -> np.ndarray:
    if abs(f - 1.0) < 1e-9:
        return img
    interp = cv2.INTER_LINEAR if f > 1.0 else cv2.INTER_AREA
    h, w = img.shape[:2]
    return cv2.resize(img, (max(1, int(round(w * f))), max(1, int(round(h * f)))),
                      interpolation=interp)


def edge_map(gray: np.ndarray, sigma: float = 1.0) -> np.ndarray:
    """Normalised gradient magnitude. Tone invariant, so good for registration."""
    g = cv2.GaussianBlur(gray, (0, 0), sigma)
    gx = cv2.Scharr(g, cv2.CV_32F, 1, 0)
    gy = cv2.Scharr(g, cv2.CV_32F, 0, 1)
    m = cv2.magnitude(gx, gy)
    hi = float(np.percentile(m[:: max(1, m.size // 200_000)], 99.5))
    return np.clip(m / (hi + EPS), 0.0, 1.0)


def edge_pair(a: np.ndarray, b: np.ndarray, mask, sigma: float = 1.0):
    """
    Gradient magnitude of two images on one shared scale.

    edge_map normalises each image by its own 99.5th percentile, which is what
    makes it tone invariant and is right when the two maps are never compared.
    Here they are subtracted, and then it is wrong: the warped reference carries
    the hard step where its coverage runs out, that step sets its percentile, and
    every real edge in the reference comes out two or three times weaker than the
    identical edge in the test. The whole silhouette of the scene then reads as a
    difference. One normaliser, measured only where the two frames overlap, is
    what makes the subtraction mean anything.
    """
    def mag(x):
        g = cv2.GaussianBlur(x, (0, 0), sigma)
        return cv2.magnitude(cv2.Scharr(g, cv2.CV_32F, 1, 0),
                             cv2.Scharr(g, cv2.CV_32F, 0, 1))
    ma, mb = mag(a), mag(b)
    va = ma[mask] if mask is not None else ma.ravel()
    vb = mb[mask] if mask is not None else mb.ravel()
    if va.size < 64:
        hi = max(float(ma.max()), float(mb.max()))
    else:
        st = max(1, va.size // 100_000)
        hi = float(np.percentile(np.concatenate([va[::st], vb[::st]]), 99.5))
    hi = max(hi, 1e-6)
    return np.clip(ma / hi, 0.0, 1.0), np.clip(mb / hi, 0.0, 1.0)


def box(img: np.ndarray, k: int) -> np.ndarray:
    k = max(1, int(k) | 1)
    return cv2.boxFilter(img, cv2.CV_32F, (k, k), normalize=True,
                         borderType=cv2.BORDER_REFLECT)


def robust_scale(x: np.ndarray, mask: np.ndarray, floor: float = 1e-5) -> tuple[float, float]:
    """
    Median and MAD-derived sigma over the masked pixels.

    `floor` must be a physically meaningful noise level for the quantity being
    measured. Without it, a pair of near-identical images drives sigma to zero
    and every rounding error becomes an enormous z-score.
    """
    v = x[mask] if mask is not None else x.ravel()
    if v.size < 64:
        return 0.0, max(floor, 1e-5)
    if v.size > 400_000:
        v = v[:: max(1, v.size // 400_000)]
    med = float(np.median(v))
    mad = float(np.median(np.abs(v - med)))
    return med, max(mad * 1.4826, floor, 1e-5)

### 3.4 Warp algebra

In [ ]:
# ============================================================================
# diffmask (1).py lines 153-185 -- warp conversion and application.
#
# Convention, and it is the one thing to get right when reading the rest: a warp W maps
# DESTINATION coordinates to SOURCE coordinates, which is what cv2.warp*(..., WARP_INVERSE_MAP)
# consumes. rewarp re-expresses W between resolutions, which is why registration can run at 640px
# and be applied at 1024px without re-solving.
# ============================================================================
# -------------------------------------------------------------------- warp math
# Convention: a warp W maps DESTINATION coordinates to SOURCE coordinates, which
# is what cv2.warp*(..., WARP_INVERSE_MAP) consumes.


def to3x3(W: np.ndarray) -> np.ndarray:
    if W.shape[0] == 3:
        return W.astype(np.float32)
    return np.vstack([W, np.array([[0.0, 0.0, 1.0]], np.float32)]).astype(np.float32)


def rewarp(W: np.ndarray, f_dst: float, f_src: float) -> np.ndarray:
    """
    Re-express W for images resampled by f_dst (destination) and f_src (source).
    Returns the same shape (2x3 or 3x3) as the input.
    """
    H = to3x3(W)
    S = np.diag([f_src, f_src, 1.0]).astype(np.float32)
    Si = np.diag([1.0 / f_dst, 1.0 / f_dst, 1.0]).astype(np.float32)
    R = (S @ H @ Si).astype(np.float32)
    R /= R[2, 2]
    return R if W.shape[0] == 3 else R[:2].copy()


def warp_into(img, W, size_wh, value=0):
    flags = cv2.INTER_LINEAR | cv2.WARP_INVERSE_MAP
    if W.shape[0] == 3:
        return cv2.warpPerspective(img, W, size_wh, flags=flags,
                                   borderMode=cv2.BORDER_CONSTANT, borderValue=value)
    return cv2.warpAffine(img, W, size_wh, flags=flags,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=value)

### 3.5 Registration

Three routes are tried in order of cost: a coarse scale and translation search on edge maps, ECC
refinement on band-passed images, and an ORB with RANSAC fallback that can also handle rotation.

`align_score` decides between them. It is edge correlation over the overlap, discounted by the
overlap itself, so a warp that keeps only a corner of the frame cannot win by correlating well on
very little.

The printed `align` value is not a measure of warp quality. It tracks how much high-contrast
detail a frame carries, so a low value on a plain surface does not mean registration failed.

In [ ]:
# ============================================================================
# diffmask (1).py lines 186-403 -- registration.
#
# Alg.1 step 1: coarse_similarity, a scale-translation template search on edge maps.
# Alg.1 step 2: ecc_refine, then feature_similarity as the rotation-capable challenger.
#
# register() ties them together. The coarse+ECC path is what runs whenever it works, which is the
# common case; the rotation paths are tried only when align_score comes back below --rot-trigger,
# and a challenger must win by --rot-margin to replace the incumbent.
# ============================================================================
# ------------------------------------------------------------------ registration


def coarse_similarity(ref_gray, test_gray, work=224, smin=0.30, smax=3.0, steps=37):
    """
    Brute force scale + translation search of reference against defect.

    Returns (s, tx, ty, ncc) describing  p_test = s * p_ref + (tx, ty).
    """
    ft = fit_long_side(test_gray.shape, work)
    t_small = edge_map(resize_f(test_gray, ft))
    r_base = edge_map(resize_f(ref_gray, ft))

    best = (1.0, 0.0, 0.0, -2.0)
    for s in np.geomspace(smin, smax, steps):
        tmpl = resize_f(r_base, float(s))
        th, tw = tmpl.shape[:2]
        if th < 8 or tw < 8 or th > 4000 or tw > 4000:
            continue
        py, px = th // 3, tw // 3  # allow overhang, but demand real overlap
        padded = cv2.copyMakeBorder(t_small, py, py, px, px, cv2.BORDER_CONSTANT, value=0.0)
        if padded.shape[0] < th or padded.shape[1] < tw:
            continue
        res = cv2.matchTemplate(padded, tmpl, cv2.TM_CCOEFF_NORMED)

        # A featureless template sitting on featureless padding correlates
        # perfectly and means nothing, so require the matched window to carry
        # a comparable amount of edge energy to the template itself.
        tmpl_energy = float(tmpl.mean())
        if tmpl_energy < 1e-4:
            continue
        ii = cv2.integral(padded)
        win = (ii[th:, tw:] - ii[:-th, tw:] - ii[th:, :-tw] + ii[:-th, :-tw]) / float(th * tw)
        res = np.where(win >= 0.35 * tmpl_energy, res, -1.0).astype(np.float32)

        _, mx, _, loc = cv2.minMaxLoc(res)
        if mx > best[3]:
            best = (float(s), (loc[0] - px) / ft, (loc[1] - py) / ft, float(mx))
    return best


def similarity_to_warp(s, tx, ty) -> np.ndarray:
    """Invert p_test = s*p_ref + t into a test -> ref warp."""
    inv = 1.0 / max(s, 1e-6)
    return np.array([[inv, 0.0, -tx * inv], [0.0, inv, -ty * inv]], np.float32)


def ecc_refine(ref_gray, test_gray, W, homography=False, work=640,
               levels=(0.25, 0.5, 1.0), iters=100, eps=1e-6):
    """
    Coarse-to-fine ECC on band-passed images, so exposure differences are
    irrelevant. W maps test -> ref, expressed at full resolution, in and out.
    """
    mode = cv2.MOTION_HOMOGRAPHY if homography else cv2.MOTION_AFFINE
    if homography:
        W = to3x3(W)

    base_t = fit_long_side(test_gray.shape, work)
    base_r = fit_long_side(ref_gray.shape, work * 2)

    for lvl in levels:
        ft, fr = base_t * lvl, base_r * lvl
        t_s = edge_map(resize_f(test_gray, ft), 1.2)
        r_s = edge_map(resize_f(ref_gray, fr), 1.2)
        if min(t_s.shape[:2]) < 32 or min(r_s.shape[:2]) < 32:
            continue
        Wl = np.ascontiguousarray(rewarp(W, ft, fr), dtype=np.float32)
        crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, iters, eps)
        try:
            _, Wl = cv2.findTransformECC(t_s, r_s, Wl, mode, crit, None, 5)
        except cv2.error:
            continue  # diverged at this level, keep what we had
        cand = rewarp(Wl, 1.0 / ft, 1.0 / fr)
        if np.isfinite(cand).all():
            W = cand.astype(np.float32)
    return W


def align_score(ref_gray, test_gray, W, work=256, min_cov=0.30):
    """
    How well a warp actually lines the pair up, as one number.

    Edge correlation over the overlap, so exposure is irrelevant, discounted by
    the overlap itself: a warp that keeps only a corner of the frame can
    correlate perfectly and still be wrong. Cheap enough (a few ms) to act as
    the referee between competing registrations.
    """
    if W is None or not np.isfinite(W).all():
        return 0.0
    f = fit_long_side(test_gray.shape, work)
    t_s = resize_f(test_gray, f)
    h, w = t_s.shape[:2]
    Ww = rewarp(W, f, 1.0)
    if not np.isfinite(Ww).all():
        return 0.0
    r_a = warp_into(ref_gray, Ww, (w, h))
    cov = warp_into(np.full(ref_gray.shape[:2], 255, np.uint8), Ww, (w, h))
    v = cv2.erode(cov, np.ones((5, 5), np.uint8)) > 0
    if int(v.sum()) < 500:
        return 0.0
    ea = edge_map(r_a)[v].ravel()
    eb = edge_map(t_s)[v].ravel()
    c = float(np.corrcoef(ea, eb)[0, 1])
    if not np.isfinite(c):
        return 0.0
    return max(0.0, c) * min(1.0, float(v.mean()) / max(min_cov, 1e-6))


def feature_similarity(ref_gray, test_gray, work=768, nfeat=3000, ratio=0.78):
    """
    Rotation-capable registration from ORB correspondences and RANSAC.

    The coarse search covers scale and translation only, by construction: it
    slides a template. Rotation needs either an extra search dimension or a
    representation that does not care, and matched keypoints are the cheap
    version of the latter, because an ORB descriptor is already rotation
    invariant. Fitted as a 4-dof similarity rather than a full affine, since
    that is the transform the pair actually differs by and the extra freedom
    only buys RANSAC ways to be confidently wrong.

    Returns (W test -> ref at full resolution, inlier count), or (None, 0).
    """
    ft = fit_long_side(test_gray.shape, work)
    fr = fit_long_side(ref_gray.shape, work)
    a = np.clip(resize_f(test_gray, ft) * 255.0, 0, 255).astype(np.uint8)
    b = np.clip(resize_f(ref_gray, fr) * 255.0, 0, 255).astype(np.uint8)
    if min(a.shape[:2]) < 40 or min(b.shape[:2]) < 40:
        return None, 0

    # Local contrast equalisation first: the pair differs in exposure, and FAST
    # corner detection is a plain intensity threshold.
    clahe = cv2.createCLAHE(2.5, (8, 8))
    a, b = clahe.apply(a), clahe.apply(b)

    orb = cv2.ORB_create(nfeat, scaleFactor=1.15, nlevels=14, fastThreshold=7,
                         edgeThreshold=19, patchSize=31)
    ka, da = orb.detectAndCompute(a, None)
    kb, db = orb.detectAndCompute(b, None)
    if da is None or db is None or len(ka) < 10 or len(kb) < 10:
        return None, 0

    bf = cv2.BFMatcher(cv2.NORM_HAMMING)
    good = []
    for pr in bf.knnMatch(da, db, k=2):
        if len(pr) == 2 and pr[0].distance < ratio * pr[1].distance:
            good.append(pr[0])
    if len(good) < 12:
        return None, 0

    pa = np.float32([ka[g.queryIdx].pt for g in good]).reshape(-1, 1, 2)
    pb = np.float32([kb[g.trainIdx].pt for g in good]).reshape(-1, 1, 2)
    M, inl = cv2.estimateAffinePartial2D(pa, pb, method=cv2.RANSAC,
                                         ransacReprojThreshold=3.0, maxIters=4000,
                                         confidence=0.995, refineIters=20)
    if M is None or not np.isfinite(M).all():
        return None, 0
    n_in = int(inl.sum()) if inl is not None else 0
    if n_in < 10:
        return None, 0
    # Reject a degenerate fit outright rather than letting ECC chase it.
    sc = float(np.hypot(M[0, 0], M[1, 0]))
    if not (0.15 < sc < 6.0):
        return None, 0
    return rewarp(M.astype(np.float32), 1.0 / ft, 1.0 / fr), n_in


def register(ref_gray, test_gray, args):
    """
    Full registration, with a rotation-capable fallback.

    The coarse-plus-ECC path stays exactly as it was and is still what runs
    whenever it works, which is the common case. Only when its own alignment
    score comes back poor are the rotation-capable paths tried, and an
    alternative has to beat the incumbent by a margin to replace it, so a pair
    that was already registered correctly cannot be talked out of it.
    """
    s, tx, ty, cscore = coarse_similarity(ref_gray, test_gray)
    W = similarity_to_warp(s, tx, ty)
    if not args.no_ecc:
        W = ecc_refine(ref_gray, test_gray, W, homography=args.homography,
                       work=args.ecc_work)
    if args.no_rot_search:
        return W, s, cscore, "coarse", -1.0

    a_best = align_score(ref_gray, test_gray, W)
    if a_best >= args.rot_trigger:
        return W, s, cscore, "coarse", a_best

    def consider(Wc, tag, best):
        if Wc is None:
            return best
        if args.homography:
            Wc = to3x3(Wc)
        cands = [Wc]
        if not args.no_ecc:
            try:
                cands.append(ecc_refine(ref_gray, test_gray, Wc,
                                        homography=args.homography, work=args.ecc_work))
            except cv2.error:
                pass
        for c in cands:
            a = align_score(ref_gray, test_gray, c)
            if a > best[0] + args.rot_margin:
                best = (a, c, tag)
        return best

    best = (a_best, W, "coarse")
    Wf, _ = feature_similarity(ref_gray, test_gray)
    best = consider(Wf, "orb", best)
    # Report the scale of the warp that won. The coarse estimate is the number a
    # user reads to sanity check registration, so it must not describe a warp
    # that was discarded.
    Wb = best[1]
    s_eff = s if best[2] == "coarse" else float(
        1.0 / max(np.sqrt(abs(np.linalg.det(to3x3(Wb)[:2, :2]))), 1e-9))
    return best[1], s_eff, cscore, best[2], best[0]

### 3.6 Dense flow, Algorithm 1 step 3

In [ ]:
# ============================================================================
# diffmask (1).py lines 404-432 -- low-frequency dense flow on the aligned reference.
#
# Deliberately computed small and smoothed hard: it must absorb slow geometric drift from a
# re-render, NOT deform itself around the defect. --flow-smooth exposes the sigma that used to be
# hardcoded here.
#
# Known limit: wallplugs are 3D objects at different depths, so each shifts differently under one
# affine warp and this is smoothed too hard to follow them. That is why image 010 needs
# --auto-low to detect at all. Lowering -k or --min-score does not help -- it surfaces four plugs
# of near-equal strength, i.e. parallax residual, not the defect.
# ============================================================================
def flow_refine(ref_aligned, test, max_px=18.0, work=320, smooth=0.06):
    """
    Low-frequency dense flow applied to the already aligned reference.

    Deliberately computed small and smoothed hard: it should absorb slow
    geometric drift from a re-render, not deform itself around the defect.
    """
    h, w = test.shape[:2]
    f = fit_long_side((h, w), work)
    a = np.ascontiguousarray(resize_f(cv2.cvtColor(ref_aligned, cv2.COLOR_BGR2GRAY), f))
    b = np.ascontiguousarray(resize_f(cv2.cvtColor(test, cv2.COLOR_BGR2GRAY), f))

    dis = cv2.DISOpticalFlow_create(cv2.DISOPTICAL_FLOW_PRESET_FAST)
    dis.setUseSpatialPropagation(True)
    flow = dis.calc(b, a, None)  # where in the reference each defect pixel comes from

    flow = cv2.GaussianBlur(flow, (0, 0), max(1.0, smooth * max(flow.shape[:2])))
    flow = cv2.resize(flow, (w, h), interpolation=cv2.INTER_LINEAR) * (1.0 / f)

    mag = cv2.magnitude(flow[..., 0], flow[..., 1])
    if (mag > max_px).any():
        flow *= np.where(mag > max_px, max_px / np.maximum(mag, EPS), 1.0).astype(np.float32)[..., None]

    gx, gy = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    out = cv2.remap(ref_aligned, gx + flow[..., 0], gy + flow[..., 1], cv2.INTER_LINEAR,
                    borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    return out, float(mag.mean())

### 3.7 Photometric matching and residuals

The pair differs in exposure as well as content, so the reference is matched to the test before
anything is subtracted. Both a whole-frame gain and offset fit and a local windowed fit are
computed, because the local fit is sharper but can erase a large flat defect.

`band_residual` then measures distance outside a tolerance band built from the reference's own
local range, rather than a plain absolute difference, so sub-pixel misregistration scores zero.

In [ ]:
# ============================================================================
# diffmask (1).py lines 433-579 -- photometric fitting and the first residual terms.
#
# Alg.1 step 4: global_match / photometric_fit build the matched reference; band_residual turns
# the comparison into a tolerance band so sub-pixel misregistration costs nothing.
# Alg.1 step 5 (part): low_freq_residual and low_freq_disagreement are the low-frequency
# appearance cue, gated behind --w-low / --auto-low.
#
# --auto-low exists because some defects only show up as a broad appearance shift with no edge:
# image 010's wallplug is the case that forced it.
# ============================================================================
# --------------------------------------------------------------------- differing


def global_match(ref_c, test_c, valid, gain=True, lo=0.5, hi=2.0):
    """
    Match the reference channel to the test channel over the whole frame.

    Unlike the local fit this cannot erase a defect, however large, because a
    single gain and offset has nowhere to hide one. Robust statistics keep the
    defect itself from dragging the fit.
    """
    mr, sr = robust_scale(ref_c, valid, 1e-3)
    mt, st = robust_scale(test_c, valid, 1e-3)
    g = float(np.clip(st / (sr + 1e-6), lo, hi)) if gain else 1.0
    return (ref_c - mr) * g + mt


def photometric_fit(ref_c, test_c, win, gain=True, lo=0.5, hi=2.0, rng=None, w=None):
    """
    Match the reference channel to the test channel on a local window.

    Note this necessarily erases any difference that is smooth across the
    window, a large flat object included. That is why the caller also keeps a
    globally matched copy and scores the low frequencies separately.
    """
    if w is None:
        mr, mt = box(ref_c, win), box(test_c, win)
    else:
        # Only pixels the two frames actually share may inform the fit. Without
        # this the window reaches past the overlap, and past the frame edge,
        # where the box filter reflects content back in: the two images then get
        # windows holding different scenes and the fit answers a question nobody
        # asked. That is why the outermost band of a re-render lights up.
        den = box(w, win) + EPS
        mr, mt = box(ref_c * w, win) / den, box(test_c * w, win) / den
    if not gain:
        out = ref_c - mr + mt
    else:
        if w is None:
            vr = np.maximum(box(ref_c * ref_c, win) - mr * mr, 0.0)
            vt = np.maximum(box(test_c * test_c, win) - mt * mt, 0.0)
        else:
            vr = np.maximum(box(ref_c * ref_c * w, win) / den - mr * mr, 0.0)
            vt = np.maximum(box(test_c * test_c * w, win) / den - mt * mt, 0.0)
        g = np.clip(np.sqrt((vt + 0.25) / (vr + 0.25)), lo, hi)
        out = mt + g * (ref_c - mr)
    # The fit is estimated from misaligned content, so near a strong edge it can
    # predict a value the channel cannot physically take. The test image then
    # disagrees with the prediction by the whole overshoot and no tolerance band
    # can forgive it, because the fitted reference holds nothing the test could
    # have matched. Clamping to the channel's own range removes that alone.
    if rng is not None:
        out = np.clip(out, rng[0], rng[1])
    return out


def band_residual(ref_c, test_c, radius):
    """
    How far the test channel falls outside the range the reference takes within
    `radius` pixels. Sub-pixel misregistration and small non-rigid drift move a
    value around inside the band and cost nothing, while genuinely new content
    has no nearby reference value to explain it.
    """
    if radius < 1:
        d = test_c - ref_c
        return np.maximum(d, 0.0) + np.maximum(-d, 0.0)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radius + 1, 2 * radius + 1))
    lo = cv2.erode(ref_c, k)
    hi = cv2.dilate(ref_c, k)
    return np.maximum(test_c - hi, 0.0) + np.maximum(lo - test_c, 0.0)


def low_freq_residual(glob, test, norm_win, tol_r):
    """
    The low frequency difference term: band residuals of the globally matched
    reference against the test, both blurred past the local fit window.

    It lives on its own because two callers have to measure the exact same
    quantity. difference_score scores it, and low_freq_disagreement decides
    whether scoring it is safe at all. If the decision and the term ever drifted
    apart, the gate would still return a number and would no longer be gating
    anything.
    """
    Lg, Ag, Bg = glob
    Lt, At, Bt = test
    lw = max(3, norm_win // 2)
    return band_residual(box(Lg, lw), box(Lt, lw), tol_r) + 0.7 * (
        band_residual(box(Ag, lw), box(At, lw), tol_r)
        + band_residual(box(Bg, lw), box(Bt, lw), tol_r))


def low_freq_disagreement(ref_bgr, test_bgr, valid, norm_win, tol_r, q=90.0, work=512):
    """
    How far apart the pair's low frequencies still are after a global fit, in
    Lab units, at the `q`th percentile over the frame.

    Measured on a downscaled copy, with both window sizes scaled to match. This
    is a statistic about low frequencies by construction, so resolution buys it
    nothing, and at working resolution it cost 188 ms of a 1000 ms run - 19% of
    the total, spent deciding whether to switch on a term that is usually left
    off anyway. Downscaling to a 512 px long side makes it ~12 ms.

    This is exactly the quantity that makes the low frequency term unsafe by
    default. That term is the only thing that can see the inside of an object
    wider than the photometric fit window, because the local fit has agreed
    with that interior and erased it; the reason it is off is that it also
    reports uneven illumination, and on a re-rendered pair the illumination is
    uneven everywhere. A percentile says which of the two is happening: a
    defect occupies a small part of the frame and leaves the percentile at the
    floor, while drifting illumination lifts the whole distribution.
    """
    f = fit_long_side(ref_bgr.shape, work)
    if f < 1.0:
        ref_bgr, test_bgr = resize_f(ref_bgr, f), resize_f(test_bgr, f)
        valid = resize_f(valid.astype(np.uint8) * 255, f) > 127
        norm_win = max(3, int(norm_win * f) | 1)
        tol_r = max(1, int(round(tol_r * f)))

    Lr, Ar, Br = cv2.split(cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2Lab))
    Lt, At, Bt = cv2.split(cv2.cvtColor(test_bgr, cv2.COLOR_BGR2Lab))
    glob = (global_match(Lr, Lt, valid, gain=True),
            global_match(Ar, At, valid, gain=False),
            global_match(Br, Bt, valid, gain=False))
    d = low_freq_residual(glob, (Lt, At, Bt), norm_win, tol_r)
    v = d[valid]
    if v.size < 1024:
        return 1e9
    return float(np.percentile(v[:: max(1, v.size // 400_000)], q))


def big_blur(img, sigma, cap=8.0):
    """
    Gaussian blur with a large sigma, evaluated on a downsampled copy.

    A kernel that wide carries no detail worth resolving at full resolution,
    and doing it directly would dominate the runtime.
    """
    if sigma <= cap:
        return cv2.GaussianBlur(img, (0, 0), sigma)
    h, w = img.shape[:2]
    f = cap / sigma
    sw, sh = max(8, int(round(w * f))), max(8, int(round(h * f)))
    small = cv2.resize(img, (sw, sh), interpolation=cv2.INTER_AREA)
    small = cv2.GaussianBlur(small, (0, 0), max(1.0, sigma * sw / float(w)))
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

### 3.8 Sharpness and the local z-score, Algorithm 1 steps 5 and 6

In [ ]:
# ============================================================================
# diffmask (1).py lines 580-680 -- the sharpness term and Z(p).
#
# sharpness_z is the fifth cue and it is DISABLED in the frozen config (--w-sharp 0). It put
# 29.9% of image 030 into the mask: a re-render is sharper or softer than its source over large
# areas for reasons that have nothing to do with a defect, so the term answers strongly and
# wrongly across whole objects.
#
# local_zscore is Eq.1 in the paper. Note it is a large-neighbourhood z-score with a variance
# floor (floor_sd), not a global threshold: a re-render is noisy in textured areas and silent on
# flat ones, so a fixed threshold is either deaf on texture or hallucinating on flat substrate.
# ============================================================================
def sharpness_z(Lr, Lt, valid, hw, agg, beta=0.35, cfloor=0.15, floor_sd=0.022,
                open_r=12, coarse=3.5):
    """
    Regional loss (or gain) of high frequency detail, in calibrated sigmas.

    A blurred patch keeps its colours and its local mean, so it is invisible to
    every other term. What it does lose is high frequency energy, and the honest
    way to ask about that is a ratio rather than a difference: the same blur
    costs 3 L-units on coarse fabric and 0.05 on a soft gradient, so an absolute
    residual is uncomparable across a frame and unthresholdable across scenes.

    Four things the earlier prototype got wrong, all measured on the photo pair:

      * it compared the high frequency maps through band_residual, whose erode
        takes the minimum of the reference over the tolerance disc. High
        frequency energy is spiky, so that minimum sits near zero almost
        everywhere and it threw away most of the signal: the true drop inside
        the patch is 0.17 L and the prototype reported 0.09, against an outside
        99th percentile of 0.11.
      * it measured against the locally gain-fitted reference, and that fit is
        driven by exactly the local variance blurring destroys, so it had
        already pulled the reference partway toward the blurred test. On the
        texture scene the fitted gain inside the patch was 0.57.
      * it scored an absolute L-unit residual against a fixed 0.15 floor. On the
        photo scenes the entire signal is 0.09, so the z-score came out below 1
        and could never have survived the thresholds downstream.
      * it was averaged into the score with the other terms, which are all silent
        on a blur, so they outvoted it three to one.

    What is measured instead: high frequency energy aggregated over a window,
    which makes it insensitive to the sub-pixel shifts a pointwise comparison
    chokes on, compared as a normalised ratio, at two scales.

    The two scales are the discriminator. A real blur is band limited: it empties
    the fine band and leaves the coarse band nearly alone. Resampling the
    reference through the warp aliases fine detail, which also empties the fine
    band, but it disturbs the coarse band by a comparable fraction because it
    moves edges rather than softening them. Subtracting the coarse ratio from the
    fine one keeps the blur and cancels the artefact. On the photo scene, whose
    packaging lettering is the worst offender, this cuts the outside 99.9th
    percentile from 0.66 to 0.44 while the inside stays three quarters of its
    value.

    The two directions are scored separately rather than folded together,
    because the artefact is not symmetric - the aliased reference reads as
    spuriously sharp - so a combined measure buries the blur direction under the
    noise of the other one. Scoring them apart also means sharpening is caught
    on its own terms and not merely as an unsigned discrepancy.
    """
    def ratio(k):
        a = box(np.abs(Lr - box(Lr, k)), k)
        b = box(np.abs(Lt - box(Lt, k)), k)
        A, B = box(a, agg), box(b, agg)
        s = A + B
        # The additive constant keeps a ratio meaningful where there is barely
        # any detail to lose. Scaled to the frame's own high frequency level so
        # it means the same thing on coarse fabric and on a soft gradient, with
        # an absolute floor for a frame that is nearly featureless.
        c = max(cfloor, beta * float(np.mean(s[valid])) if valid.any() else cfloor)
        return (A - B) / (s + c)

    r_fine = ratio(hw)
    r_coarse = ratio(max(9, int(coarse * hw) | 1))

    # `floor_sd` is not a safety net here, it is the calibration. The ratio is
    # dimensionless and bounded by one, so what counts as a real loss of detail
    # is a fixed fraction and not whatever the MAD of this particular frame
    # happens to be. Left to the MAD the scale collapses: on a frame that
    # matches almost everywhere the MAD is zero, and a ratio of 0.09, which is
    # nothing, came out at 22 sigmas and swallowed the whole image.
    out = []
    for sgn in (1.0, -1.0):
        d = np.maximum(np.maximum(sgn * r_fine, 0.0)
                       - np.maximum(sgn * r_coarse, 0.0), 0.0)
        med, sig = robust_scale(d, valid, floor_sd)
        out.append(np.clip((d - med) / max(sig, floor_sd), 0.0, None))

    # A sharpness defect is a region. Anything narrower than the window the
    # measurement was made over is not a measurement of sharpness at all, it is
    # an edge that moved, and on the photo scene that means every letter of the
    # packaging text. A grey opening removes exactly those: a plateau wider than
    # the element keeps its value, a ridge thinner than it drops to its
    # surroundings.
    if open_r >= 1:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * int(open_r) + 1,) * 2)
        out = [cv2.morphologyEx(x, cv2.MORPH_OPEN, k) for x in out]
    # Returned as two channels, lost detail and gained detail, because the
    # caller treats them differently.
    return cv2.merge([out[0].astype(np.float32), out[1].astype(np.float32)])


def local_zscore(score, valid, sigma, floor_sd):
    """Z-score against a large local neighbourhood, ignoring invalid pixels."""
    v = valid.astype(np.float32)
    den = big_blur(v, sigma) + EPS
    m = big_blur(score * v, sigma) / den
    m2 = big_blur(score * score * v, sigma) / den
    sd = np.sqrt(np.maximum(m2 - m * m, 0.0))
    return (score - m) / np.maximum(sd, floor_sd)

### 3.9 Fusing the score

Combines luminance, chromaticity, gradient, low-frequency appearance and sharpness into a single
difference map. `--combine-p` controls the combination and defaults to a plain average.

Two consequences worth knowing. A defect that differs in only one channel is outvoted by the terms
blind to it. And `--smooth` blurs the score, so a defect narrower than the blur radius loses most
of its peak while broader background noise survives.

Both are levers for thin defect recovery, and both change behaviour on every category.

In [ ]:
# ============================================================================
# diffmask (1).py lines 681-796 -- difference_score, the fusion of Alg.1 step 5.
#
# Returns (S, dvec, z_sharp). dvec is the per-channel signed Lab difference and it matters later:
# complete_object grows a region by matching the DIRECTION of dvec, which is how a partially
# detected defect is completed without also swallowing whatever else is nearby.
#
# The term weights (--w-luma/--w-chroma/--w-grad/--w-low/--w-sharp) were function arguments only
# until they were exposed as CLI flags. They are left at their defaults here -- reweighting them
# per category is exactly the category-specific tuning supplementary 3.A says FLASH does not do.
# ============================================================================
def difference_score(ref_bgr, test_bgr, valid, norm_win, tol_r, smooth=0.45,
                     w_luma=1.0, w_chroma=0.7, w_grad=0.9,
                     w_low=0.0, w_sharp=0.0, p=1.0,
                     fit_clamp=False, fit_mask=False, shared_edge=False,
                     sharp_win=21, sharp_beta=0.35, sharp_sd=0.022, sharp_open=12):
    """
    Photometric-invariant dissimilarity map, in units of robust sigmas.

    Every term is a band residual computed after a local photometric fit, so
    exposure drift and small geometric error are absorbed rather than reported.
    """
    Lr, Ar, Br = cv2.split(cv2.cvtColor(ref_bgr, cv2.COLOR_BGR2Lab))
    Lt, At, Bt = cv2.split(cv2.cvtColor(test_bgr, cv2.COLOR_BGR2Lab))

    # Globally matched copies, which keep large flat differences intact. Only
    # built when something actually needs them.
    Lg = Ag = Bg = None
    if w_low > 0:
        Lg = global_match(Lr, Lt, valid, gain=True)
        Ag = global_match(Ar, At, valid, gain=False)
        Bg = global_match(Br, Bt, valid, gain=False)

    # Kept before the local fit, because the sharpness term below must not see a
    # reference whose contrast has already been pulled toward the blurred test.
    Lr_raw = Lr

    # Locally matched copies, which are immune to uneven tone but blind to
    # anything smooth across the fit window.
    rL, rC = ((0.0, 100.0), (-128.0, 127.0)) if fit_clamp else (None, None)
    fw_ = valid.astype(np.float32) if fit_mask else None
    Lr = photometric_fit(Lr, Lt, norm_win, gain=True, rng=rL, w=fw_)
    Ar = photometric_fit(Ar, At, norm_win, gain=False, rng=rC, w=fw_)
    Br = photometric_fit(Br, Bt, norm_win, gain=False, rng=rC, w=fw_)

    d_luma = band_residual(Lr, Lt, tol_r)
    d_chroma = band_residual(Ar, At, tol_r) + band_residual(Br, Bt, tol_r)

    if shared_edge:
        gr, gt = edge_pair(Lr / 100.0, Lt / 100.0, valid, 1.0)
    else:
        gr, gt = edge_map(Lr / 100.0, 1.0), edge_map(Lt / 100.0, 1.0)
    d_grad = band_residual(gr, gt, tol_r)

    # Low frequencies, scored on the globally matched copies. This is what sees
    # an object bigger than the local fit window, whose interior the local fit
    # has quietly agreed with. Off by default: it also sees uneven illumination.
    d_low = None
    if w_low > 0:
        d_low = low_freq_residual((Lg, Ag, Bg), (Lt, At, Bt), norm_win, tol_r)

    # Local high frequency energy. A region that was blurred, or sharpened,
    # keeps its colours and its local mean and shows up in nothing else.
    # Computed as a z-score in its own right and folded in by the caller with a
    # maximum, not averaged in as one more term: it is a specialist that is
    # silent on every other defect, and an average would let the silent
    # majority veto it.
    z_sharp = None
    if w_sharp > 0:
        z_sharp = sharpness_z(Lr_raw, Lt, valid, max(3, tol_r | 1),
                              max(5, int(sharp_win) | 1), sharp_beta,
                              floor_sd=sharp_sd, open_r=sharp_open)

    # Floors are in each term's own units, and set the smallest difference
    # worth calling real.
    terms = [(d_luma, w_luma, 0.25), (d_chroma, w_chroma, 0.40),
             (d_grad, w_grad, 0.010)]
    if d_low is not None:
        terms.append((d_low, w_low, 0.25))

    # Combined as a power mean. p=1 is the plain average and is the default,
    # because it is what the thresholds downstream are calibrated against.
    # Raising p moves toward a maximum, which suits a mix of specialist terms:
    # a translucent film answers in chroma and gradient and nowhere else, a
    # large flat object only in the low frequencies, a blurred patch only in
    # sharpness. Under an average the silent terms outvote the one term that
    # can actually see the defect, and each term added makes that worse.
    total_w, acc = 0.0, None
    for arr, wgt, floor in terms:
        if wgt <= 0:
            continue
        med, sig = robust_scale(arr, valid, floor)
        z = np.clip((arr - med) / sig, 0.0, None)
        contrib = wgt * (z if p == 1.0 else np.power(z, p))
        acc = contrib if acc is None else acc + contrib
        total_w += wgt

    score = acc / max(total_w, 1e-6)
    if p != 1.0:
        score = np.power(score, 1.0 / p)
    if smooth > 0:
        score = cv2.GaussianBlur(score, (0, 0), max(0.6, tol_r * smooth))
    score[~valid] = 0.0

    # Returned alongside the score rather than mixed into it, so that every
    # statistic the caller derives from the score - its global median, its
    # global sigma, its local mean and spread - is bit for bit what it was
    # before this term existed. The caller folds it in at the point of
    # thresholding, which is the only place it can add anything.
    if z_sharp is not None:
        z_sharp = cv2.GaussianBlur(z_sharp, (0, 0), max(0.6, tol_r * 0.5))
        z_sharp = (w_sharp * z_sharp).astype(np.float32)
        z_sharp[~valid] = 0.0

    # The signed Lab difference is returned too. Its magnitude is what scores
    # above, but its direction is what identifies an object, and the two parts
    # of one object agree on direction long after the fainter part has dropped
    # below any magnitude threshold. Measured against the local fit normally,
    # which isolates the object's own effect; against the global fit when the
    # low frequency term is on, since the local fit erases a large interior.
    if Lg is None:
        dvec = cv2.merge([Lt - Lr, At - Ar, Bt - Br])
    else:
        dvec = cv2.merge([Lt - Lg, At - Ag, Bt - Bg])
    return score.astype(np.float32), dvec, z_sharp

### 3.10 Morphology, hysteresis and completion, Algorithm 1 steps 7 to 9

In [ ]:
# ============================================================================
# diffmask (1).py lines 797-1009 -- from thresholded Z to final components.
#
# Alg.1 step 7: hysteresis grows each strong seed through the connected weak region around it,
# so faint defect extensions survive while isolated weak noise does not.
# Alg.1 step 8: select_components ranks by accumulated evidence and keeps by --keep-ratio /
# --top-k / --min-area.
# Alg.1 step 9: complete_object recovers weak continuations by matching the direction of the
# difference vector rather than its magnitude.
#
# shrink_keep_thin is the --shrink implementation and it is the one to watch. At the stock 1.2 it
# shattered image 015 into 13 fragments -- 166px surviving from an 1835px seed -- because a
# uniform erosion is fatal to any region narrower than twice the radius. 0.4 is the frozen value.
#
# --keep-ratio 0.50 drops a thin region even once it is the strongest candidate, because a thin
# region's MEAN strength is diluted by its own length. Left at 0.50 for every category; that is
# a known cost of one configuration for all eight, and it surfaces as an EMPTY mask in section 4.
# ============================================================================
def hysteresis(strong, weak, max_grow=12.0):
    """
    Grow each strong seed through the connected weak region around it.

    A translucent object only differs strongly at its edges, so seeding on the
    strong evidence and growing through weak evidence recovers the whole shape.
    A component that balloons far beyond its seed is treated as a leak and
    falls back to the seed.
    """
    weak_u8 = weak.astype(np.uint8)
    n, lab = cv2.connectedComponents(weak_u8, connectivity=8)
    if n <= 1:
        return strong.astype(np.uint8) * 255

    seed_area = np.bincount(lab[strong].ravel(), minlength=n).astype(np.float64)
    full_area = np.bincount(lab.ravel(), minlength=n).astype(np.float64)
    keep = seed_area > 0
    keep[0] = False
    keep &= full_area <= max_grow * np.maximum(seed_area, 1.0)

    out = keep[lab]
    return (out | strong).astype(np.uint8) * 255


def fill_holes(mask):
    """Fill background regions fully enclosed by the mask."""
    h, w = mask.shape
    inv = cv2.copyMakeBorder(cv2.bitwise_not(mask), 1, 1, 1, 1,
                             cv2.BORDER_CONSTANT, value=255)
    # Foreground is treated as 8-connected, so the background must be 4-connected
    # for a hole to count as enclosed.
    n, lab = cv2.connectedComponents(inv, connectivity=4)
    if n <= 1:
        return mask
    outside = lab[0, 0]
    holes = ((lab != outside) & (inv > 0))[1:h + 1, 1:w + 1]
    out = mask.copy()
    out[holes] = 255
    return out


def within(mask, radius):
    """Pixels no further than `radius` from the mask. A big structuring element
    would do the same thing, at many times the cost."""
    if radius < 1:
        return mask > 0
    d = cv2.distanceTransform((mask == 0).astype(np.uint8), cv2.DIST_L2, 3)
    return d <= radius


def shrink_keep_thin(mask, radius):
    """
    Pull every boundary in by `radius` without amputating slender parts.

    The distance transform gives each pixel's local half width, so eroding is
    just `dist > radius`. Anything narrower than that would vanish entirely, so
    those parts keep their medial ridge, detected as a local maximum of the
    distance transform.
    """
    dist = cv2.distanceTransform(mask, cv2.DIST_L2, 3)
    eroded = dist > radius
    if not eroded.any():
        return mask

    # Thin means narrower than the shrink, so the erosion would lose it whole.
    # Judged per part rather than per component, so a slender tail on a solid
    # body is treated as thin even though its component survives.
    thin = (mask > 0) & ~within(eroded.astype(np.uint8) * 255, radius)

    out = eroded.astype(np.uint8) * 255
    if thin.any():
        ridge = (dist >= cv2.dilate(dist, np.ones((3, 3), np.float32)) - 1e-4) & (dist > 1.5)
        out |= (ridge & (mask > 0)).astype(np.uint8) * 255
        out = cv2.morphologyEx(out, cv2.MORPH_CLOSE,
                               cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
        out = cv2.bitwise_and(out, mask)
        out[eroded] = 255
    return out


def object_edges(test_bgr, lo=10, hi=30, thicken=True):
    """
    Silhouette of whatever is in the defect image, tuned to be over-inclusive.

    The asymmetry matters: a spurious edge only stops the growth below early,
    while a missing one lets it escape the object entirely. So this errs
    sensitive, and the result is thickened so barriers are watertight.
    """
    gray = cv2.GaussianBlur(cv2.cvtColor(test_bgr, cv2.COLOR_BGR2GRAY), (0, 0), 1.2)
    edges = cv2.Canny(gray.astype(np.uint8), lo, hi)
    if thicken:
        edges = cv2.dilate(edges, np.ones((3, 3), np.uint8))
    return edges


def complete_object(mask, dvec, valid, radius, frac=0.30, noise_k=3.0,
                    max_grow=5.0, edges=None):
    """
    Grow each region into the rest of the same object.

    Differencing responds where an object differs most, which for a translucent
    one is its densest part. A shape tapering to a point barely differs from
    what it covers near the tip, so the tip is missed however low a magnitude
    threshold is set, and growing on magnitude alone just bleeds into whatever
    else happens to be noisy.

    Direction is the discriminator that survives. One object changes the image
    the same way throughout, only more faintly where it is thin, so projecting
    the local difference onto the region's own mean direction separates the rest
    of that object from unrelated texture by a wide margin. The measured margin
    on translucent plastic over jelly is about 3.5 against 0.0.

    `edges`, when given, stops growth crossing a hard silhouette.
    """
    # Fragments lying close together are parts of one object, not separate
    # objects. Grouping them first matters: thin extremities often survive
    # detection only as a dotted line, and each dot on its own is both too small
    # to estimate a signature from and disconnected from the body it belongs to.
    grp = within(mask, max(1, radius // 2)).astype(np.uint8)
    n, glab, stats, _ = cv2.connectedComponentsWithStats(grp, connectivity=8)
    if n <= 1:
        return mask

    H, W = mask.shape
    out = np.zeros_like(mask)
    pad = int(2.0 * radius) + 4  # growth zone, plus a ring to measure noise in

    for i in range(1, n):
        # Everything below is local to the object, so crop to its neighbourhood
        # rather than sweeping the whole frame once per object.
        bx, by, bw, bh = stats[i, :4]
        x0, y0 = max(0, bx - pad), max(0, by - pad)
        x1, y1 = min(W, bx + bw + pad), min(H, by + bh + pad)

        seed = (glab[y0:y1, x0:x1] == i) & (mask[y0:y1, x0:x1] > 0)
        seed_area = int(seed.sum())
        if seed_area == 0:
            continue
        d = dvec[y0:y1, x0:x1]

        sig = d[seed].mean(axis=0)
        norm = float(np.linalg.norm(sig))
        if norm < 1e-6:
            out[y0:y1, x0:x1][seed] = 255
            continue
        unit = sig / norm

        sub_valid = valid[y0:y1, x0:x1]
        reachable = within(seed.astype(np.uint8) * 255, radius) & sub_valid
        proj = d @ unit

        # Measure noise in the ring just outside the growth zone: near enough to
        # describe this part of the image, far enough that the missing part of
        # the object cannot inflate it and threshold itself away.
        med, sd = robust_scale(proj, sub_valid & ~reachable, 1e-3)
        thr = max(frac * float(proj[seed].mean()), med + noise_k * sd)

        cand = ((proj > thr) & reachable) | seed
        if edges is not None:
            cand &= (edges[y0:y1, x0:x1] == 0) | seed
        cand = cv2.morphologyEx(cand.astype(np.uint8), cv2.MORPH_CLOSE,
                                cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))

        nc, lc = cv2.connectedComponents(cand, connectivity=8)
        touched = np.unique(lc[seed & (cand > 0)])
        keep = np.zeros(nc, bool)
        keep[touched[touched != 0]] = True
        grown = keep[lc] | seed

        out[y0:y1, x0:x1][grown if grown.sum() <= max_grow * seed_area else seed] = 255

    return fill_holes(out)


def select_components(binary, min_area, keep_ratio, score, k, top_k=0):
    """
    Rank regions by total evidence above threshold, not by area alone.

    Also returns the regions that cleared --min-area and were then dropped for
    being weaker than keep_ratio times the best one. That rule assumes the frame
    holds one dominant defect, and when it does not - two defects of unequal
    strength, or anything spurious that outranks the real one - it discards the
    answer without saying so. Measured on fabric 134, whose reference carries a
    speck of lint the defect frame does not: the speck scores 1.000 and the
    thread that is actually the defect scores 0.201, so the thread was dropped
    and the mask sat on blank fabric. Handing the dropped regions back lets the
    caller draw them, which is the difference between a wrong answer and a
    wrong answer you can see.
    """
    n, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    if n <= 1:
        return np.zeros_like(binary), 0, [], np.zeros_like(binary)
    excess = np.clip(score - k, 0.0, None)
    cand = []
    for i in range(1, n):
        area = int(stats[i, cv2.CC_STAT_AREA])
        if area < min_area:
            continue
        cand.append((float(excess[labels == i].sum()), i, area))
    if not cand:
        return np.zeros_like(binary), 0, [], np.zeros_like(binary)

    cand.sort(reverse=True)
    top = cand[0][0]
    keep = cand[:top_k] if top_k > 0 else [c for c in cand if c[0] >= keep_ratio * top]
    kept_ids = {i for _, i, _ in keep}
    out, dropped = np.zeros_like(binary), np.zeros_like(binary)
    for _, i, _ in cand:
        (out if i in kept_ids else dropped)[labels == i] = 255
    report = [(a, s / max(top, 1e-9), stats[i, :4].tolist()) for s, i, a in cand]
    return out, len(keep), report, dropped

### 3.11 `run()`, the pipeline end to end

In [ ]:
# ============================================================================
# diffmask (1).py lines 1010-1292 -- run(), the whole pipeline end to end.
#
# Reads the pair, registers, flows, scores, thresholds, completes, writes the mask and optionally
# the overlay and the 7 debug intermediates. Returns 0.
#
# This is the function the paper's Algorithm 1 describes. Everything above is what it calls.
# ============================================================================
# -------------------------------------------------------------------------- main


def run(args) -> int:
    t0 = time.perf_counter()
    ref_full = imread(args.reference)
    test_full = imread(args.defect)
    Ht, Wt = test_full.shape[:2]

    ref_g = to_gray32(ref_full)
    test_g = to_gray32(test_full)

    t_reg = time.perf_counter()
    W, s, cscore, reg_mode, _ = register(ref_g, test_g, args)
    t_reg = time.perf_counter() - t_reg

    # ---- render the reference into the defect frame at working resolution --
    fw = fit_long_side((Ht, Wt), args.work)
    test_w = resize_f(test_full, fw)
    hw, ww = test_w.shape[:2]

    Ww = rewarp(W, fw, 1.0)
    ref_aligned = warp_into(ref_full, Ww, (ww, hw))
    cov = warp_into(np.full(ref_full.shape[:2], 255, np.uint8), Ww, (ww, hw))

    # Re-renders and rescales always disagree along the frame edge, so drop both
    # the seam where the warped reference runs out and a margin of the frame
    # itself. cv2.erode leaves the image border alone, hence the explicit trim.
    b = int(args.border * max(hw, ww))
    er = max(3, b * 2 + 1)
    valid = cv2.erode(cov, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (er, er)))
    if b > 0:
        valid[:b, :] = 0
        valid[-b:, :] = 0
        valid[:, :b] = 0
        valid[:, -b:] = 0

    flow_px = 0.0
    if not args.no_flow:
        ref_aligned, flow_px = flow_refine(ref_aligned, test_w, max_px=args.flow_max * fw,
                                           smooth=args.flow_smooth)
        alive = (cv2.cvtColor(ref_aligned, cv2.COLOR_BGR2GRAY) > 0).astype(np.uint8) * 255
        valid = cv2.bitwise_and(valid, cv2.erode(alive, np.ones((5, 5), np.uint8)))

    # A clipped pixel holds no information. Whatever was there is gone, the other
    # frame has nothing to be compared against, and the photometric fit cannot
    # recover it either since there is nothing left to fit. Measured on the jelly
    # pairs, whose reference blows out along the cup's right wall while the defect
    # frame still resolves it: the wall then reads as new content for its whole
    # length. No weighting of the terms removes that, because the difference is
    # real in pixels and meaningless in fact - which is exactly the case validity
    # is for. Both frames are tested, since either one clipping is enough to make
    # the comparison empty.
    if args.sat_guard > 0:
        lim = args.sat_guard * 255.0
        sat = ((ref_aligned.min(axis=2) >= lim) | (test_w.min(axis=2) >= lim)).astype(np.uint8)
        sat = cv2.dilate(sat, np.ones((5, 5), np.uint8))  # resampling spreads it
        valid[sat > 0] = 0

    valid_b = valid > 0
    if valid_b.mean() < 0.05:
        print("diffmask: warning: registration overlap is tiny, result is unreliable",
              file=sys.stderr)

    # Every stage downstream assumes the two frames actually correspond. When
    # they do not, the output is not a smaller mask, it is a meaningless one, so
    # say so rather than returning quiet nonsense.
    ea = edge_map(to_gray32(ref_aligned))[valid_b].ravel()
    eb = edge_map(to_gray32(test_w))[valid_b].ravel()
    step = max(1, ea.size // 200_000)
    align = float(np.corrcoef(ea[::step], eb[::step])[0, 1]) if ea.size > 1000 else 0.0
    if not np.isfinite(align):
        align = 0.0
    if align < args.min_align:
        print(f"diffmask: warning: the two images align poorly (edge correlation "
              f"{align:.2f}, expected above {args.min_align:.2f}). every stage after "
              f"this assumes they show the same objects. if they show different "
              f"instances of the same kind of object, or differ by a rotation the "
              f"coarse search does not cover, then everything differs and no mask "
              f"here means anything. treat the output as unreliable.", file=sys.stderr)

    # ---- difference --------------------------------------------------------
    L = max(hw, ww)
    norm_win = max(5, int(args.norm_frac * L) | 1)
    tol_r = max(1, int(round(args.tol_frac * L)))
    ref_d = ref_aligned.astype(np.float32) / 255.0
    test_d = test_w.astype(np.float32) / 255.0

    # The low frequency term is the only one that can see the middle of an
    # object wider than the fit window, and it is off by default because it
    # also reports uneven illumination. Measure whether this pair has any, and
    # switch the term on only when it does not.
    w_low, lf_q = args.w_low, -1.0
    if args.auto_low > 0 and args.w_low <= 0:
        lf_q = low_freq_disagreement(ref_d, test_d, valid_b, norm_win, tol_r)
        if lf_q <= args.auto_low:
            w_low = args.auto_low_w

    score, dvec, z_sharp = difference_score(
        ref_d, test_d, valid_b, norm_win, tol_r, args.smooth,
        w_luma=args.w_luma, w_chroma=args.w_chroma, w_grad=args.w_grad,
        w_low=w_low, w_sharp=args.w_sharp, p=args.combine_p,
        fit_clamp=not args.no_fit_clamp,
        fit_mask=not args.no_fit_mask,
        shared_edge=not args.no_shared_edge,
        sharp_win=max(5, int(args.sharp_frac * L) | 1),
        sharp_beta=args.sharp_beta, sharp_sd=args.sharp_sd,
        sharp_open=int(args.sharp_open * L))

    # The re-render noise floor varies a lot across the frame, so judge every
    # pixel against its own neighbourhood rather than against a global constant.
    med, sig = robust_scale(score, valid_b, 0.10)
    z = local_zscore(score, valid_b, max(4.0, args.adapt_frac * L), max(0.5 * sig, 0.10))

    # Folded in here, after med, sig and the local statistics have all been
    # taken from the untouched score, so nothing the sharpness term does can
    # move an existing detection. Only into z, which is already in sigmas: the
    # score carries the other terms' units, and mixing a sharpness sigma into
    # it would make -k and --abs-k retune this term as a side effect.
    if z_sharp is not None:
        # Only the gained-detail half stands back where the ordinary terms
        # already have an answer. This term is measured over a window and then
        # opened, so it locates a defect to within about thirty pixels and no
        # better. That is fine for a blurred patch, which nothing else sees at
        # all. It is not fine for a three pixel fibre: a fibre is new detail, so
        # it reads as a local gain in sharpness, and folding that in inflates
        # the line into a thirty pixel band and throws away the precision the
        # other terms had earned. The lost-detail half is not restrained the
        # same way, because losing detail is not something adding an object
        # does, so it has no thin defect to smear.
        lost, gained = z_sharp[..., 0], z_sharp[..., 1]
        if args.sharp_avoid > 0:
            near = within((z > args.k).astype(np.uint8) * 255,
                          int(args.sharp_avoid * L))
            gained = np.where(near, 0.0, gained).astype(np.float32)
        z_sharp = np.maximum(lost, gained)
        z = np.maximum(z, z_sharp)
    z[~valid_b] = 0.0

    # ---- threshold, at full resolution -------------------------------------
    z_full = cv2.resize(z, (Wt, Ht), interpolation=cv2.INTER_CUBIC)
    score_full = cv2.resize(score, (Wt, Ht), interpolation=cv2.INTER_CUBIC)
    valid_full = cv2.resize(valid, (Wt, Ht), interpolation=cv2.INTER_NEAREST) > 0
    # An absolute floor as well as a relative one, so a pair that simply does
    # not differ anywhere yields an empty mask instead of amplified noise.
    thr = max(med + args.abs_k * sig, args.min_score)
    abs_ok = score_full > thr
    weak_abs = score_full > thr * args.low_ratio
    # The sharpness channel clears the absolute floor on its own scale instead
    # of on `thr`, which is built from the other terms' median and sigma. A
    # sharpness sigma and a score are different quantities, and letting this
    # term borrow thr made -k, --abs-k and --min-score silently retune blur
    # sensitivity while being changed for unrelated reasons.
    if z_sharp is not None:
        zs_full = cv2.resize(z_sharp, (Wt, Ht), interpolation=cv2.INTER_CUBIC)
        abs_ok |= zs_full > args.sharp_floor
        weak_abs |= zs_full > args.sharp_floor * args.low_ratio
    floor_ok = abs_ok & valid_full
    strong = (z_full > args.k) & floor_ok
    weak = (z_full > args.k * args.low_ratio) & weak_abs & valid_full

    r = max(3, int(0.004 * max(Ht, Wt)) | 1)
    strong = cv2.morphologyEx(strong.astype(np.uint8), cv2.MORPH_OPEN,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (r, r))) > 0

    binary = hysteresis(strong, weak, args.max_grow)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (r * 3, r * 3)))
    binary = fill_holes(binary)

    # Decide which regions are defects before touching their boundaries. Doing
    # it the other way round lets the shrink below sever a slender part, which
    # then gets discarded here as a separate weak region.
    min_area = max(24, int(args.min_area * Ht * Wt))
    mask, n_kept, allc, dropped = select_components(
        binary, min_area, args.keep_ratio, z_full, args.k * args.low_ratio, args.top_k)

    # The tolerance band and the score smoothing both widen every boundary by a
    # known amount, so take that much back.
    shrink = int(round(tol_r * args.shrink / max(fw, 1e-6)))
    if shrink >= 1 and mask.any():
        mask = shrink_keep_thin(mask, shrink)

    # Complete the shape, so a tapering end that barely differs from what it
    # covers still comes out whole.
    # With the low frequency term active the mask arrives as a sparse core of a
    # much larger shape, since a wide object differs most at its rim and its
    # near-uniform interior is what the local noise estimate flattens hardest,
    # so the completion is given a wider radius and a far looser area bound.
    snap_r, snap_grow = args.snap, args.snap_grow
    if w_low > 0:
        snap_r, snap_grow = args.low_snap, args.low_snap_grow
    if snap_r > 0 and mask.any():
        d_full = cv2.resize(dvec, (Wt, Ht), interpolation=cv2.INTER_LINEAR)
        edges_full = object_edges(test_full, args.canny_lo, args.canny_hi) \
            if args.snap_edges else None
        mask = complete_object(mask, d_full, valid_full,
                               max(2, int(snap_r * max(Ht, Wt))),
                               args.snap_frac, args.snap_noise, snap_grow,
                               edges_full)

    if args.dilate:
        d = max(1, int(args.dilate)) | 1
        mask = cv2.dilate(mask, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (d, d)))

    # select_components already enforced --min-area, but two stages have run
    # since: shrink_keep_thin, which can sever a region into ridge fragments,
    # and complete_object, which can split one and leave debris behind. Neither
    # is filtered, so single pixel regions reach the output - rice 126 shipped
    # two of them. The rule is applied a second time here rather than moved,
    # because the reason it runs before the shrink in the first place (a severed
    # slender arm must not then be discarded as a separate weak region) still
    # holds.
    if mask.any():
        n, lab, st, _ = cv2.connectedComponentsWithStats((mask > 0).astype(np.uint8), 8)
        drop = np.zeros(max(n, 1), bool)
        for i in range(1, n):
            drop[i] = st[i, cv2.CC_STAT_AREA] < min_area
        if drop.any():
            mask[drop[lab]] = 0

    imwrite(args.out, mask)
    dt = time.perf_counter() - t0

    # Count what was actually written. n_kept is decided before the completion
    # stage, which can split a region or leave fragments behind, so reporting it
    # describes an intermediate the user never sees.
    area = int((mask > 0).sum())
    n_out = cv2.connectedComponents((mask > 0).astype(np.uint8), connectivity=8)[0] - 1
    print(f"register  scale={s:.4f} coarse_ncc={cscore:.3f} align={align:.2f} "
          f"via={reg_mode} "
          f"{'homography' if args.homography else 'affine'} "
          f"flow={flow_px:.2f}px  {t_reg * 1000:.0f}ms")
    print(f"lowfreq   low_freq_p90={lf_q:.2f} w_low={w_low:.2f}")
    print(f"threshold median={med:.3f} sigma={sig:.3f} z>{args.k} and score>{thr:.3f}")
    n_drop = cv2.connectedComponents((dropped > 0).astype(np.uint8), connectivity=8)[0] - 1
    print(f"mask      {n_out} region(s) {area}px "
          f"({100.0 * area / (Ht * Wt):.3f}% of image) -> {args.out}"
          + (f"  [{n_kept} before completion]" if n_out != n_kept else ""))
    if n_drop:
        # Said out loud rather than left to -v: a dropped region is the shape a
        # missed defect takes, and it costs nothing to mention.
        print(f"dropped   {n_drop} region(s) {int((dropped > 0).sum())}px below "
              f"--keep-ratio {args.keep_ratio}, outlined in red on the overlay")
    if args.verbose:
        for j, (a, rel, bb) in enumerate(allc):
            print(f"  [{'x' if j < n_kept else ' '}] strength={rel:.3f} "
                  f"raw_area={a:7d} bbox={bb}")
    print(f"total     {dt * 1000:.0f} ms")

    if args.overlay or args.debug:
        # Outline only by default: a filled overlay hides the very thing you are
        # trying to check the boundary against.
        ov = test_full.copy()
        if area and args.overlay_fill > 0:
            f = min(max(args.overlay_fill, 0.0), 1.0)
            ov[mask > 0] = ((1 - f) * ov[mask > 0] + f * np.array([0, 0, 255])).astype(np.uint8)
        # Discarded candidates first, so a kept region drawn over one stays
        # readable. Red is not "a second detection", it is "this cleared every
        # bar except being half as strong as the winner".
        if not args.no_show_dropped and dropped.any():
            dc, _ = cv2.findContours(dropped, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
            cv2.drawContours(ov, dc, -1, (0, 0, 255), args.overlay_width)
        cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(ov, cnts, -1, (255, 255, 0), args.overlay_width)
        if args.overlay:
            imwrite(args.overlay, ov)
            print(f"overlay   -> {args.overlay}")

    if args.debug:
        d = Path(args.debug)
        imwrite(str(d / "01_ref_aligned.png"), ref_aligned)
        imwrite(str(d / "02_test_work.png"), test_w)
        imwrite(str(d / "03_blend.png"), cv2.addWeighted(ref_aligned, 0.5, test_w, 0.5, 0))
        sn = np.clip(score / max(float(score.max()), 1e-6) * 255, 0, 255).astype(np.uint8)
        imwrite(str(d / "04_score.png"), cv2.applyColorMap(sn, cv2.COLORMAP_TURBO))
        zn = np.clip(z / max(args.k, 1e-6) * 160, 0, 255).astype(np.uint8)
        imwrite(str(d / "04b_zscore.png"), cv2.applyColorMap(zn, cv2.COLORMAP_TURBO))
        imwrite(str(d / "05_overlay.png"), ov)
        imwrite(str(d / "06_valid.png"), valid)
    return 0

### 3.12 The argument parser

Defines the command line interface, so all available flags are documented here and the batch cell
below can call `main(argv)` with the same defaults.

The `if __name__ == "__main__"` guard from the original file is deliberately absent. In a notebook
`__name__` is `"__main__"`, so it would run against the kernel's own arguments.

In [ ]:
# ============================================================================
# diffmask (1).py lines 1293-1470 -- main(), the argument parser.
#
# The __main__ guard (lines 1472-1474) is deliberately excluded: __name__ == "__main__" is TRUE
# in a notebook, so it would fire on cell execution.
#
# main(argv) both parses and runs -- the last line is `return run(p.parse_args(argv))` -- which is
# exactly how run_diffmask_batch.py drives it. The notebook does the same rather than
# reconstructing an argparse.Namespace by hand, so no default can silently diverge.
# ============================================================================
def main(argv=None) -> int:
    p = argparse.ArgumentParser(
        prog="diffmask",
        description="Binary mask of the object present only in the defect image.")
    p.add_argument("reference", help="clean reference image")
    p.add_argument("defect", help="image containing the extra object")
    p.add_argument("-o", "--out", default="mask.png", help="output mask path")
    p.add_argument("--overlay", metavar="PNG", help="also write the mask outlined on the defect image")
    p.add_argument("--overlay-fill", type=float, default=0.0,
                   help="tint the interior by this much, 0 = outline only (default)")
    p.add_argument("--overlay-width", type=int, default=2, help="outline thickness in px")
    p.add_argument("--debug", metavar="DIR", help="write intermediate images here")

    p.add_argument("--work", type=int, default=1024, help="working long side (default 1024)")
    p.add_argument("--ecc-work", type=int, default=640, help="registration long side")
    p.add_argument("--homography", action="store_true", help="8-dof warp instead of affine")
    p.add_argument("--no-ecc", action="store_true", help="skip ECC refinement")
    p.add_argument("--no-flow", action="store_true", help="skip dense flow refinement")
    p.add_argument("--flow-max", type=float, default=18.0, help="flow cap, in output px")
    p.add_argument("--flow-smooth", type=float, default=0.06,
                   help="how hard the dense flow is smoothed, as a fraction of "
                        "the flow field's own long side. the default is "
                        "deliberately heavy so the flow absorbs global drift "
                        "without deforming around the defect. lower it when the "
                        "scene is several rigid objects at different depths, "
                        "where each one shifts differently and one affine warp "
                        "cannot follow them; the cost is that a large soft "
                        "defect starts being absorbed too")
    p.add_argument("--no-rot-search", action="store_true",
                   help="never try the rotation-capable registration fallback")
    p.add_argument("--rot-trigger", type=float, default=0.60,
                   help="try the rotation fallback when the coarse+ECC warp "
                        "scores below this alignment")
    p.add_argument("--rot-margin", type=float, default=0.02,
                   help="alignment the fallback must gain before it replaces the "
                        "coarse warp")
    p.add_argument("--min-align", type=float, default=0.75,
                   help="warn below this edge correlation between the aligned pair. "
                        "measured: 0.92 and 0.96 on pairs that differ by one added "
                        "object, 0.59 on a pair whose objects are different objects")
    p.add_argument("--sat-guard", type=float, default=0.99,
                   help="treat a pixel as invalid when either frame is clipped "
                        "at or above this fraction of full scale in every "
                        "channel. a blown highlight has lost what was there, so "
                        "the frames cannot be compared and any difference found "
                        "is an artefact of one of them clipping first. 0 disables")
    p.add_argument("--border", type=float, default=0.015,
                   help="ignore this fraction of the long side around the overlap seam")

    p.add_argument("--norm-frac", type=float, default=0.04,
                   help="photometric fit window, as a fraction of the long side")
    p.add_argument("--tol-frac", type=float, default=0.006,
                   help="geometric tolerance band radius, fraction of the long side")
    p.add_argument("--adapt-frac", type=float, default=0.15,
                   help="local noise estimation radius, fraction of the long side")
    p.add_argument("--smooth", type=float, default=0.60,
                   help="score smoothing, as a multiple of the tolerance radius")
    p.add_argument("--w-luma", type=float, default=1.0,
                   help="weight of the L band residual")
    p.add_argument("--w-chroma", type=float, default=0.7,
                   help="weight of the a+b band residual")
    p.add_argument("--w-grad", type=float, default=0.9,
                   help="weight of the gradient band residual. this is the term "
                        "a thin high-contrast structure answers in, so lower it "
                        "when the frame contains one that registration cannot "
                        "settle - a specular rim on transparent plastic being "
                        "the case it was measured on")
    p.add_argument("--w-low", type=float, default=0.0,
                   help="weight for the low frequency term; needed for defects "
                        "wider than the fit window, but also sees uneven light")
    p.add_argument("--auto-low", type=float, default=0.0,
                   help="turn the low frequency term on by itself when the "
                        "pair's low frequencies already agree to within this "
                        "many Lab units over 90%% of the frame (0 = never). "
                        "2.5 is the calibrated value and is what to pass to "
                        "enable this; it is off by default because the term is "
                        "a low pass, so it widens every boundary it touches by "
                        "its own radius, which measured -0.13 IoU on a real "
                        "pair whose defect is smaller than the fit window")
    p.add_argument("--auto-low-w", type=float, default=0.6,
                   help="weight the low frequency term gets when --auto-low fires")
    p.add_argument("--w-sharp", type=float, default=1.0,
                   help="gain on the sharpness term; needed for blur or "
                        "sharpening defects, which no other term can see. NOTE "
                        "this is not the averaged-in band residual it named "
                        "before: it is a maximum-folded two-scale ratio, folded "
                        "in after every other statistic has been taken. 0 "
                        "disables it. it is not purely additive: raising z also "
                        "widens the hysteresis growth of regions the other "
                        "terms found, so existing detections can change shape. "
                        "it resolves to about 30px, so it is a regional change "
                        "detector, not only a blur detector")
    p.add_argument("--sharp-frac", type=float, default=0.021,
                   help="window the sharpness term aggregates over, as a "
                        "fraction of the long side")
    p.add_argument("--sharp-beta", type=float, default=0.35,
                   help="sharpness ratio softening, as a fraction of the "
                        "frame's own mean high frequency energy")
    p.add_argument("--sharp-avoid", type=float, default=0.05,
                   help="ignore GAINED-detail sharpness evidence within this "
                        "fraction of the long side of a region the other terms "
                        "already found, since they localise it far better")
    p.add_argument("--sharp-open", type=float, default=0.012,
                   help="drop sharpness evidence narrower than this fraction "
                        "of the long side; a blur is a region, a moved edge is "
                        "a filament")
    p.add_argument("--sharp-sd", type=float, default=0.022,
                   help="what counts as one sigma of sharpness ratio; this is "
                        "a calibration, not a floor, because the ratio is "
                        "dimensionless. it sets the scale that -k and "
                        "--sharp-floor are then read in, so lowering it makes "
                        "the term more sensitive at both of those gates at once")
    p.add_argument("--sharp-floor", type=float, default=4.0,
                   help="absolute gate the sharpness term must clear, in its "
                        "own sigmas; the other terms use --min-score and "
                        "--abs-k, which are not in these units")
    p.add_argument("--no-shared-edge", action="store_true",
                   help="normalise each gradient map by its own percentile "
                        "instead of one shared over the overlap")
    p.add_argument("--no-fit-mask", action="store_true",
                   help="let the local photometric fit read pixels outside the "
                        "overlap and outside the frame")
    p.add_argument("--no-fit-clamp", action="store_true",
                   help="let the local photometric fit predict values outside "
                        "the channel range")
    p.add_argument("--combine-p", type=float, default=1.0,
                   help="power mean exponent, 1 = average; raise toward 3 when "
                        "extra terms are enabled so specialists are not outvoted")
    p.add_argument("--shrink", type=float, default=1.20,
                   help="boundary shrink, as a multiple of the tolerance radius")
    p.add_argument("--snap", type=float, default=0.035,
                   help="complete each region to the whole object within this "
                        "fraction of the long side (0 disables)")
    p.add_argument("--snap-frac", type=float, default=0.30,
                   help="keep pixels matching this fraction of the region's own signature")
    p.add_argument("--snap-noise", type=float, default=2.0,
                   help="floor for the completion threshold, in robust sigmas")
    p.add_argument("--snap-grow", type=float, default=5.0,
                   help="max area the completion may add, relative to the seed")
    p.add_argument("--low-snap", type=float, default=0.08,
                   help="completion radius used instead of --snap while the "
                        "low frequency term is active")
    p.add_argument("--low-snap-grow", type=float, default=120.0,
                   help="completion area allowance used instead of --snap-grow "
                        "while the low frequency term is active")
    p.add_argument("--snap-edges", action="store_true",
                   help="also stop completion at hard silhouette edges")
    p.add_argument("--canny-lo", type=int, default=20, help="edge detector low threshold")
    p.add_argument("--canny-hi", type=int, default=60, help="edge detector high threshold")
    p.add_argument("-k", type=float, default=5.0, help="threshold in local sigmas")
    p.add_argument("--abs-k", type=float, default=3.0,
                   help="absolute floor in global robust sigmas")
    p.add_argument("--min-score", type=float, default=4.0,
                   help="hard minimum score, so matching pairs return an empty mask")
    p.add_argument("--low-ratio", type=float, default=0.35,
                   help="hysteresis growth threshold, relative to -k")
    p.add_argument("--max-grow", type=float, default=12.0,
                   help="max area a region may gain relative to its seed")
    p.add_argument("--min-area", type=float, default=2e-4,
                   help="min region area as a fraction of the image")
    p.add_argument("--keep-ratio", type=float, default=0.50,
                   help="keep regions this strong relative to the best one")
    p.add_argument("--top-k", type=int, default=0,
                   help="keep exactly the N strongest regions (0 = use --keep-ratio)")
    p.add_argument("--dilate", type=int, default=0, help="grow the final mask by N px")
    p.add_argument("--no-show-dropped", action="store_true",
                   help="do not outline the regions --keep-ratio discarded. by "
                        "default the overlay draws them in red, because that "
                        "rule silently throws away a region whenever something "
                        "else in the frame outranks it, and when the winner is "
                        "spurious the answer disappears with no trace. on "
                        "fabric 134 the mask sits on a speck of lint that is on "
                        "the reference and not on the defect frame, while the "
                        "thread that is the actual defect scored 0.201 and was "
                        "dropped. the mask is unchanged either way - this only "
                        "decides whether you can see what was discarded")
    p.add_argument("-v", "--verbose", action="store_true", help="list every candidate region")
    return run(p.parse_args(argv))

## 4. Batch extraction

Recovers a mask for every pair and writes it beside the anomaly frame it came from:

```
Defect Masks and Anomalies/<category>/<id>_mask.png
Defect Masks and Anomalies/<category>/<id>_anomaly.png
```

Each pair prints its runtime and the recovered mask size. Watch for two markers. `EMPTY` means
nothing was recovered from that pair, which is a result rather than an error. `FAIL` means the
call raised; the batch continues so one bad pair does not cost the rest.

Debug intermediates are written to `_work/`. One of them, the registered normal image, is used in
section 7, so this is not optional.

In [ ]:
import io as _io, contextlib, traceback

RESULTS = []   # one row per pair

for cat, pid, ref, dfc in PAIRS:
    # The recovered mask is written straight into the deliverable folder, beside a copy of the
    # anomaly frame it came from. That pair IS the Stage 2 output; nothing else needs keeping.
    odir = os.path.join(PAIRS_DIR, cat); os.makedirs(odir, exist_ok=True)
    mask_p = os.path.join(odir, f"{pid}_mask.png")
    shutil.copy2(dfc, os.path.join(odir, f"{pid}_anomaly.png"))

    # --debug is always on now, and not for debugging: 01_ref_aligned.png is the NORMAL image
    # registered into I_a's frame, which is what section 7 shows VLM-2 as the before-panel.
    # Without it the model is asked to spot an anomaly from a single image, which is not a
    # question that can be answered. The files land in _work/, which is not a deliverable.
    dbg_p = os.path.join(WORK_DIR, cat, f"{pid}_debug")
    argv = [ref, dfc, "-o", mask_p, "--debug", dbg_p, "-v", *FLAGS]
    if WRITE_DEBUG:
        argv += ["--overlay", os.path.join(WORK_DIR, cat, f"{pid}_overlay.png")]
    buf, t0, err = _io.StringIO(), time.time(), None
    try:
        with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
            rc = main(argv)                     # diffmask's own entry point, unmodified
    except SystemExit as e:                     # imread failure calls sys.exit
        rc, err = int(e.code or 1), "SystemExit"
    except Exception as e:
        rc, err = 1, f"{type(e).__name__}: {e}"
        buf.write("\n" + traceback.format_exc())
    secs = time.time() - t0

    m = cv2.imread(mask_p, cv2.IMREAD_GRAYSCALE) if os.path.exists(mask_p) else None
    px = int((m > 127).sum()) if m is not None else 0
    RESULTS.append(dict(cat=cat, pid=pid, ref=ref, defect=dfc, mask=mask_p, dbg=dbg_p,
                        rc=rc, err=err, secs=secs, mask_px=px,
                        log=buf.getvalue()))
    print(f"{cat:13s} {pid:>6s}  {secs:5.1f}s  {px:7d}px"
          f"{'  FAIL ' + str(err) if err else ''}"
          f"{'  EMPTY' if err is None and px == 0 else ''}")

_ok    = [r for r in RESULTS if r["err"] is None and r["mask_px"] > 0]
_empty = [r for r in RESULTS if r["err"] is None and r["mask_px"] == 0]
_fail  = [r for r in RESULTS if r["err"] is not None]
print(f"\n{len(_ok)}/{len(RESULTS)} recovered a non-empty mask "
      f"| {len(_empty)} empty | {len(_fail)} failed "
      f"| {sum(r['secs'] for r in RESULTS):.0f}s total")
if _empty:
    print("empty (the mask is the label -- an empty one means no usable defect was recovered):")
    for r in _empty:
        print(f"  {r['cat']}/{r['pid']}")

An empty mask means no usable defect was recovered from that pair. It contributes no bank entry
and no ground truth, and is counted separately from a crash.

## 5. Crop extraction

Turns each recovered region into a bank candidate: a square crop centred on the defect, at
`CTX_EXPAND` times its bounding box, so the crop carries its own surrounding substrate.

The substrate is required. Module 3 matches the crop's own background against the host's before
compositing, and a cut-out with no background cannot be matched. For the same reason the defect
mask is stored as a separate file rather than as an alpha channel.

The overlays below draw every candidate region on the frame it came from. Look at them. Anything
here that is not a defect has to be rejected by the gates in sections 7 and 8.

In [ ]:
from scipy import ndimage
from PIL import Image

def load_u8(path):
    return np.array(Image.open(path).convert("RGB"))

def load_mask_u8(path):
    return np.array(Image.open(path).convert("L"))


# Rebuild flash_part1's RAW structure from what section 4 wrote, so regions_of and
# extract_entry below can be used exactly as they are in that notebook.
RAW, ids = {}, []
for r in _ok:
    k = f"{r['cat']}-{r['pid']}"                 # ids are NOT globally unique across categories
    # ref_aligned is the NORMAL image warped into I_a's own frame by diffmask's registration.
    # I_n and I_a are NOT aligned as they sit on disk -- different scale and offset -- so the raw
    # normal cannot be cropped at the same coordinates. The registered one can.
    _ra = os.path.join(r["dbg"], "01_ref_aligned.png")
    RAW[k] = dict(cat=r["cat"], did=r["pid"],
                  anomaly=load_u8(r["defect"]),
                  mask=load_mask_u8(r["mask"]),
                  normal=load_u8(r["ref"]),
                  ref_aligned=(load_u8(_ra) if os.path.exists(_ra) else None),
                  secs=r["secs"])
    ids.append(k)

_n_ra = sum(1 for k in ids if RAW[k]["ref_aligned"] is not None)
print(f"registered normal available for {_n_ra}/{len(ids)} donors"
      + ("" if _n_ra == len(ids) else "  <- the rest fall back to a 2-panel prompt"))


# --- Region filtering (verbatim from flash_part1) -----------------------------------
def regions_of(mask_u8, min_px=MIN_REGION_PX):
    """Connected components of a recovered mask, largest first, speckle removed."""
    lbl, n = ndimage.label(mask_u8 > 127)
    out = []
    for k in range(1, n + 1):
        b = lbl == k
        a = int(b.sum())
        if a < min_px:
            continue
        ys, xs = np.where(b)
        out.append(dict(bin=b, area=a, x=int(xs.min()), y=int(ys.min()),
                        w=int(xs.max() - xs.min() + 1), h=int(ys.max() - ys.min() + 1)))
    return sorted(out, key=lambda r: -r["area"])


# flash_part1 filters known false positives here with a hand-maintained DROP_REGIONS blacklist.
# There is no blacklist in this notebook -- that is exactly the job VLM-2 does in section 7.
KEPT = {i: regions_of(RAW[i]["mask"]) for i in ids}
print(f"{'donor':20s} {'cat':13s} kept  dropped-speckle")
for i in ids:
    allr = regions_of(RAW[i]["mask"], min_px=1)
    print(f"{i:20s} {RAW[i]['cat']:13s} {len(KEPT[i]):4d}  {len(allr)-len(KEPT[i]):15d}   "
          + "  ".join(f"a={r['area']}({r['w']}x{r['h']})" for r in KEPT[i]))
print(f"\n{sum(len(v) for v in KEPT.values())} candidate regions from {len(ids)} donors")

In [ ]:
# --- THE OVERLAY. Every recovered region drawn on its own anomaly frame. -------------
# This replaces diffmask's --overlay file: same information, rendered at a size you can judge,
# and one panel per REGION rather than one per image, which is the unit the bank is built from.
# Adapted from flash_part1's region-QA figure.
import matplotlib.pyplot as plt

_qa = [(i, r) for i in ids for r in KEPT[i]]
if _qa:
    _ncol = 6
    _nrow = int(math.ceil(len(_qa) / _ncol))
    fig, ax = plt.subplots(_nrow, _ncol, figsize=(2.6 * _ncol, 2.9 * _nrow), squeeze=False)
    for k, (i, r) in enumerate(_qa):
        pad = int(0.6 * max(r["w"], r["h"])) + 12
        H, W = RAW[i]["mask"].shape[:2]
        y0, y1 = max(0, r["y"] - pad), min(H, r["y"] + r["h"] + pad)
        x0, x1 = max(0, r["x"] - pad), min(W, r["x"] + r["w"] + pad)
        crop = np.ascontiguousarray(RAW[i]["anomaly"][y0:y1, x0:x1].copy())
        cnts, _ = cv2.findContours(r["bin"][y0:y1, x0:x1].astype(np.uint8),
                                   cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(crop, cnts, -1, (0, 255, 0), 2)
        a = ax[divmod(k, _ncol)]
        a.imshow(crop)
        a.set_title(f"{i}\n a={r['area']} ({r['w']}x{r['h']})", fontsize=8)
    for a in ax.ravel():
        a.axis("off")
    plt.tight_layout(); plt.show()
    print("Every candidate region, contoured on the frame it was recovered from.")
    print("Anything here that is NOT a defect is what VLM-2 has to reject in section 7.")
else:
    print("no regions survived MIN_REGION_PX -- nothing to show")

In [ ]:
# --- The recovered masks themselves, whole-frame, next to their anomaly --------------
# The per-region view above cannot show a mask that landed somewhere absurd. This can.
_show = ids[:8]
fig, ax = plt.subplots(len(_show), 3, figsize=(11, 3.6 * len(_show)), squeeze=False)
for r_i, i in enumerate(_show):
    d = RAW[i]
    ov = d["anomaly"].copy()
    cnts, _ = cv2.findContours((d["mask"] > 127).astype(np.uint8),
                               cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(ov, cnts, -1, (255, 0, 0), 3)
    ax[r_i, 0].imshow(d["anomaly"]);          ax[r_i, 0].set_title(f"{i} - I_a", fontsize=9)
    ax[r_i, 1].imshow(d["mask"], cmap="gray"); ax[r_i, 1].set_title("M_d recovered", fontsize=9)
    ax[r_i, 2].imshow(ov);                     ax[r_i, 2].set_title("overlay", fontsize=9)
for a in ax.ravel():
    a.axis("off")
plt.tight_layout(); plt.show()
print(f"First {len(_show)} of {len(ids)} donors. This is the pair written to")
print(f"  {PAIRS_DIR}/<category>/")

In [ ]:
# --- Bank extraction (verbatim from flash_part1) ------------------------------------
def extract_entry(donor_id, region, ctx=CTX_EXPAND, soften=ALPHA_SOFTEN):
    """Substrate-retained square crop + soft defect alpha + scale metadata."""
    d = RAW[donor_id]
    anom = d["anomaly"]; m = region["bin"]
    H, W = anom.shape[:2]
    dh, dw = region["h"], region["w"]
    side = int(np.clip(round(max(dh, dw) * ctx), 16, min(H, W)))
    cy, cx = region["y"] + dh // 2, region["x"] + dw // 2
    ty = int(np.clip(cy - side // 2, 0, H - side))
    tx = int(np.clip(cx - side // 2, 0, W - side))

    rgb   = anom[ty:ty + side, tx:tx + side].copy()
    alpha = m[ty:ty + side, tx:tx + side].astype(np.float32).copy()
    if soften > 0:
        alpha = cv2.GaussianBlur(alpha, (0, 0), soften)
    alpha = np.clip(alpha, 0, 1)

    # Internal contrast: how far the masked pixels sit from this crop's own substrate, in the
    # same Lab units as DEFECT_T. An entry whose mask covers plain substrate scores ~0 here, and
    # it is worse than useless -- replayed, it produces an image whose ground truth labels a
    # region where nothing changed, so every metric is scored against a defect that is not there.
    # That is what a mis-recovered diffmask looks like, and it has to be caught before the bank.
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    dsel, ssel = alpha > 0.5, alpha < 0.1
    if dsel.sum() >= 25 and ssel.sum() >= 25:
        ref  = lab[ssel].mean(0)
        dist = np.linalg.norm(lab[dsel] - ref, axis=1)
        contrast = float(np.median(dist))
        cfrac    = float((dist > DEFECT_T).mean())
    else:
        contrast, cfrac = 0.0, 0.0

    return dict(
        key=f"{donor_id}_{region['x']}_{region['y']}", donor=donor_id, cat=d["cat"],
        rgb=rgb, alpha=alpha, side=side,
        defect_long_px=int(max(dh, dw)),
        defect_frac=float(max(dh, dw) / max(H, W)),
        area_px=region["area"], area_frac=float(region["area"] / m.size),
        contrast=contrast, cfrac=cfrac,
        kind=DEFECT_KIND[d["cat"]],
    )


CROPS = []
for i in ids:
    for r in KEPT[i]:
        e = extract_entry(i, r)
        e["_region"] = r                 # kept for section 7's panels; not serialised
        e["substrate_frac"] = float((e["alpha"] < 0.1).mean())
        e["diffmask_secs"] = float(RAW[i]["secs"])
        e["flags"] = " ".join(FLAGS)
        CROPS.append(e)

# Rank each donor's regions by area so VLM-2 can be told "this is region k of n". The rank is
# a prior, not a decision -- the largest component is usually but not always the real defect.
_by_donor = {}
for e in CROPS:
    _by_donor.setdefault(e["donor"], []).append(e)
for _d, _rs in _by_donor.items():
    _rs.sort(key=lambda e: -e["area_px"])
    for _k, _e in enumerate(_rs):
        _e["n_regions"] = len(_rs)
        _e["region_rank"] = _k + 1

_multi = {d: r for d, r in _by_donor.items() if len(r) > 1}
print(f"{len(CROPS)} candidate crops from {len(ids)} masks")
if _multi:
    print(f"{len(_multi)} donor(s) returned more than one region -- at most one can be the "
          f"introduced defect:")
    for d, rs in sorted(_multi.items()):
        print(f"  {d:22s} " + "  ".join(f"{e['key'].split('_',1)[1]}({e['area_px']}px)"
                                        for e in rs))
print()
print(f"{'entry key':30s} {'cat':13s} {'crop':>9s} {'defect':>8s} {'frac':>7s}"
      f" {'subst%':>7s} {'contrast':>9s} {'cfrac':>6s}")
for e in CROPS:
    print(f"{e['key']:30s} {e['cat']:13s} {e['side']:7d}px {e['defect_long_px']:6d}px"
          f" {e['defect_frac']:7.4f} {100*e['substrate_frac']:6.1f}%"
          f" {e['contrast']:9.1f} {e['cfrac']:6.2f}")

In [ ]:
# --- Look at them. A bank you have not seen is a bank you cannot defend. -------------
import matplotlib.pyplot as plt

_n = min(len(CROPS), 12)
if _n:
    fig, ax = plt.subplots(3, _n, figsize=(1.7 * _n, 5.6))
    ax = np.atleast_2d(ax)
    for j, e in enumerate(CROPS[:_n]):
        ax[0, j].imshow(e["rgb"]);                ax[0, j].set_title(e["key"][-16:], fontsize=7)
        ax[1, j].imshow(e["alpha"], cmap="gray"); ax[1, j].set_title("alpha (stored apart)", fontsize=7)
        ring = np.zeros((*e["alpha"].shape, 3), np.float32)
        ring[..., 1] = e["alpha"]                 # green = defect
        ring[..., 2] = (e["alpha"] < 0.10)        # blue  = substrate the harmoniser samples
        ax[2, j].imshow(ring)
        ax[2, j].set_title(f"dE {e['contrast']:.0f} / cf {e['cfrac']:.2f}", fontsize=7)
        for r in range(3):
            ax[r, j].axis("off")
    plt.tight_layout(); plt.show()
    print("Green = recovered defect. Blue = the substrate Stage 5's harmoniser measures.")
    print("A crop with almost no blue will place badly however good its mask is.")

## 6. Load the validation model

Loads the same model as Module 1, in fp16 across both T4s.

If the processor fails to load, the cell falls through several routes and finally builds it from
its parts, then recovers the chat template. This handles mounts missing
`preprocessor_config.json`.

In [ ]:
import torch, re
from PIL import Image
from transformers import AutoProcessor

try:
    from transformers import Qwen2_5_VLForConditionalGeneration as _VLModel
except ImportError:
    # transformers v5 moved the class; the generic auto class resolves it from the config.
    from transformers import AutoModelForImageTextToText as _VLModel


def find_model():
    """Use MODEL_ID if it exists on disk, else search /kaggle/input for the weights."""
    if MODEL_ID and os.path.isdir(MODEL_ID):
        return MODEL_ID
    if MODEL_ID:
        print(f"  MODEL_ID not on disk ({MODEL_ID}) -- searching instead")
    hits = []
    for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
        d = os.path.dirname(cfg)
        if not glob.glob(os.path.join(d, "*.safetensors")):
            continue                                   # config without weights -> skip
        try:
            arch = json.load(open(cfg)).get("architectures", [""])[0]
        except Exception:
            arch = ""
        if "Qwen2_5_VL" in arch or "qwen" in d.lower():
            hits.append((d, arch))
    if not hits:
        raise FileNotFoundError("no mounted Qwen model under /kaggle/input -- attach it, "
                                "or set MODEL_ID to a hub id")
    hits.sort(key=lambda t: t[0])
    for d, a in hits:
        print("  model candidate:", d, "|", a)
    return hits[-1][0]


MODEL_PATH = find_model()
print("model:", MODEL_PATH)

torch.manual_seed(SEED)
_kw = dict(attn_implementation="sdpa",     # Turing: no flash-attn
           device_map="auto")              # split the 7B across both T4s
if LOAD_4BIT:
    from transformers import BitsAndBytesConfig
    _kw["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)

_t0 = time.time()
# v5 renamed torch_dtype -> dtype. Try the new name, fall back to the old one.
try:
    model = _VLModel.from_pretrained(MODEL_PATH, dtype=torch.float16, **_kw).eval()
except TypeError:
    model = _VLModel.from_pretrained(MODEL_PATH, torch_dtype=torch.float16, **_kw).eval()


# ---------------------------------------------------------------- processor
# Kaggle model mounts predate transformers v5. v5 resolves the image processor from
# `image_processor_type` in preprocessor_config.json and NO LONGER falls back to config.json's
# `model_type`, so a 4.x checkpoint raises "Unrecognized image processor" even though the
# weights loaded fine. The cascade below tries the normal routes, then patches a copy of the
# small config files, then finally builds the processor FROM ITS PARTS -- which bypasses
# config-driven class resolution altogether and is the route that works when the rest do not.
print("files in model dir:", sorted(os.listdir(MODEL_PATH)))
_pcf = os.path.join(MODEL_PATH, "preprocessor_config.json")
if os.path.exists(_pcf):
    try:
        print("preprocessor_config.json keys:", sorted(json.load(open(_pcf)).keys()))
    except Exception as e:
        print("preprocessor_config.json unreadable:", e)
else:
    print("preprocessor_config.json: ABSENT")


def _image_processor_cls():
    """The image-processor class that actually exists in THIS transformers build.

    Hardcoding "Qwen2_5_VLImageProcessor" into a patched config is wrong when the installed
    version only ships Qwen2VLImageProcessor -- Qwen2.5-VL reuses Qwen2-VL's image processor in
    several releases. Resolve the name at runtime rather than guessing it.
    """
    import transformers
    for n in ("Qwen2_5_VLImageProcessor", "Qwen2VLImageProcessor"):
        if hasattr(transformers, n):
            return n, getattr(transformers, n)
    return None, None


def _video_processor_cls():
    """v5 wants a video processor even for image-only use. Absent in 4.x, which is fine."""
    import transformers
    for n in ("Qwen2_5_VLVideoProcessor", "Qwen2VLVideoProcessor"):
        if hasattr(transformers, n):
            return n, getattr(transformers, n)
    return None, None


IP_NAME, IP_CLS = _image_processor_cls()
VP_NAME, VP_CLS = _video_processor_cls()
print(f"image processor class: {IP_NAME or 'NONE FOUND'} | video: {VP_NAME or 'none'}")


def _patched_config_dir(path, dst="/kaggle/working/_qwen_processor"):
    """Copy only the small config/tokenizer files to a writable dir and inject the v5 keys.

    The mount is read-only, hence the copy -- weights are NOT copied, only json/txt (a few MB).
    The destination is cleared first: a half-written dir left by an earlier failed attempt
    would otherwise be picked up and fail again for a different reason.
    """
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    os.makedirs(dst, exist_ok=True)
    for f in os.listdir(path):
        if f.endswith((".json", ".txt")) and f != "model.safetensors.index.json":
            shutil.copy2(os.path.join(path, f), os.path.join(dst, f))

    pc = os.path.join(dst, "preprocessor_config.json")
    try:
        cfg = json.load(open(pc)) if os.path.exists(pc) else {}
    except Exception:
        cfg = {}
    if IP_NAME:
        cfg["image_processor_type"] = IP_NAME    # set, not setdefault: overwrite a stale value
    cfg.setdefault("processor_class", "Qwen2_5_VLProcessor")
    if VP_NAME:
        cfg.setdefault("video_processor_type", VP_NAME)
    json.dump(cfg, open(pc, "w"), indent=2)
    return dst


def load_processor(path):
    errors = []

    for kw in (dict(min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS, use_fast=False),
               dict(use_fast=False)):                    # 1) straight AutoProcessor
        try:
            return AutoProcessor.from_pretrained(path, **kw)
        except Exception as e:
            errors.append(f"AutoProcessor({', '.join(kw)}): {type(e).__name__}: {e}")

    try:                                                 # 2) patched config dir
        pdir = _patched_config_dir(path)
        proc = AutoProcessor.from_pretrained(pdir, min_pixels=MIN_PIXELS,
                                             max_pixels=MAX_PIXELS, use_fast=False)
        print(f"  loaded via patched config dir ({pdir}) -- "
              "checkpoint predates the v5 image_processor_type key")
        return proc
    except Exception as e:
        errors.append(f"patched dir: {type(e).__name__}: {e}")

    try:                                                 # 3) concrete class, patched dir
        from transformers import Qwen2_5_VLProcessor
        return Qwen2_5_VLProcessor.from_pretrained(_patched_config_dir(path))
    except Exception as e:
        errors.append(f"Qwen2_5_VLProcessor: {type(e).__name__}: {e}")

    try:                                                 # 4) assemble from parts
        # The route that survives when the config cannot be resolved at all: every class is
        # instantiated directly, so nothing ever reads image_processor_type.
        from transformers import AutoTokenizer, Qwen2_5_VLProcessor
        if IP_CLS is None:
            raise ImportError("no Qwen image processor class in this transformers build")
        tok = AutoTokenizer.from_pretrained(path)
        try:
            ip = IP_CLS(min_pixels=MIN_PIXELS, max_pixels=MAX_PIXELS)
        except TypeError:
            ip = IP_CLS()                                # older signature; caps applied below
        kw = dict(image_processor=ip, tokenizer=tok)
        if VP_CLS is not None:
            try:
                kw["video_processor"] = VP_CLS()
            except Exception:
                pass                                     # image-only use; not fatal
        print("  built processor from tokenizer + image processor")
        return Qwen2_5_VLProcessor(**kw)
    except Exception as e:
        errors.append(f"manual build: {type(e).__name__}: {e}")

    raise RuntimeError("could not build a processor:\n  " + "\n  ".join(errors))


processor = load_processor(MODEL_PATH)

# Apply the pixel caps if they did not go in through the constructor.
_ip = getattr(processor, "image_processor", None)
if _ip is not None:
    for _k, _v in (("min_pixels", MIN_PIXELS), ("max_pixels", MAX_PIXELS)):
        try:
            setattr(_ip, _k, _v)
        except Exception:
            pass

# A hand-built processor has NO chat template, and vlm() calls apply_chat_template in the very
# next cell -- so recover it from the mount or the tokenizer before anything else runs.
if not getattr(processor, "chat_template", None):
    _ct = None
    _ctf = os.path.join(MODEL_PATH, "chat_template.json")
    if os.path.exists(_ctf):
        try:
            _ct = json.load(open(_ctf)).get("chat_template")
        except Exception:
            _ct = None
    if _ct is None:
        _ct = getattr(getattr(processor, "tokenizer", None), "chat_template", None)
    if _ct:
        processor.chat_template = _ct
        print("  chat template recovered")
    else:
        print("  WARNING: no chat template found -- apply_chat_template may fail")

LOAD_SECONDS = time.time() - _t0
# Reported separately everywhere below. It is paid once per session, not per crop, so folding
# it into a per-crop figure would misstate Stage 3's cost -- same convention as Stage 1.
print(f"loaded in {LOAD_SECONDS:.0f}s | 4bit={LOAD_4BIT}")
print("device map:", getattr(model, "hf_device_map", "single device"))

In [ ]:
def vlm(image, system, question, max_new_tokens=MAX_NEW_TOKENS, temperature=0.0):
    """One VLM call: PIL image + question -> (raw text reply, generate seconds).

    The timer starts only AFTER the image has been preprocessed and moved to the GPU, so the
    figure is the time Qwen spends looking and answering -- not tokenisation. Stage 3's cost
    is a paper claim, so it has to be measured the same way Stage 1 measures its own.
    """
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": [{"type": "image"},
                                         {"type": "text", "text": question}]}]
    text   = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
    kw = dict(max_new_tokens=max_new_tokens, do_sample=temperature > 0)
    if temperature > 0:
        kw.update(temperature=temperature, top_p=0.9)

    if torch.cuda.is_available():
        torch.cuda.synchronize()          # do not time work still queued from before
    t0 = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, **kw)
    if torch.cuda.is_available():
        torch.cuda.synchronize()          # generate is async; wait before stopping the clock
    secs = time.time() - t0

    reply = processor.decode(out[0][inputs.input_ids.shape[1]:],
                             skip_special_tokens=True).strip()
    return reply, secs


def grab_json(raw):
    """Pull the first JSON object out of a reply, tolerating stray prose or code fences."""
    m = re.search(r"[\[{].*[\]}]", raw, re.S)
    if not m:
        raise ValueError("no JSON in reply")
    return json.loads(m.group(0))

## 7. Semantic validation

Each candidate crop is shown to the model, which decides whether the outlined region is a genuine
defect or an artifact of extraction.

The model sees three panels: the same view before the edit, after it, and with the candidate
region outlined. It is asked what changed inside the outline first, and only then whether that
change is a defect. Comparing against a before image is what makes the question answerable; from a
single image, normal texture and a real defect look alike.

It returns JSON: whether anything changed, what changed, whether it is a defect, a name for it, a
confidence, a reason, and an artifact class. The verdict gates the crop; the name becomes the
`defect_type` column; the artifact class and reason explain rejections.

When several regions come from one image, the model is told the count and the rank, because
Module 1 introduces exactly one defect per image and the rest are extraction artifacts.

Set `VLM_MIN_CONF` and `USE_VLM_CONFIDENCE` in section 1 if you want confidence to gate as well as
the verdict.

In [ ]:
# NEUTRAL. An earlier version of this prompt told the model that most regions are not defects
# and that "no change" is the answer to reach for first. That is a thumb on the scale, and on a
# 7B model it collapsed the accept rate -- trading the original false-accept problem for a
# false-reject one. State the task and the failure modes; do not state a base rate.
SYS_VLM2 = (
    "You are a quality-control inspector for industrial visual inspection. "
    "You are shown a BEFORE and an AFTER photograph of the same product, plus a candidate "
    "region. Your job is to say what changed inside that region, and whether the change is a "
    "genuine manufacturing defect or an artifact of the imaging and extraction process. "
    "Judge only on the evidence in the two photographs. Both answers are equally acceptable: "
    "report a defect when you can see one, and report no change when the two photographs look "
    "the same inside the outline. "
    "Reply with a single JSON object and no other text."
)

# Three-panel prompt. The before-panel is what makes this answerable: "is this walnut crevice
# damage or normal shell?" cannot be decided from one image, and a model forced to decide it
# anyway falls back on "does this look irregular", which is why plain texture was being accepted.
Q_VLM2_3 = """Panel 1 (LEFT) is a reference photograph of a {cat} BEFORE any change.
Panel 2 (MIDDLE) is the SAME view of the SAME object AFTER an edit was applied.
Panel 3 (RIGHT) is panel 2 with a candidate region outlined in red.
{multi}
Work in this order:

STEP 1. Compare panel 1 and panel 2 INSIDE the outlined area only. State what is different.
        If they look the same there, the answer is no change -- say so and stop.
STEP 2. Only if something changed, decide whether that change is a genuine defect on a {cat}
        ({kind_h}), or one of these extraction artifacts:
          - normal product texture, grain, weave or print that shifted between the two shots
          - a shadow, highlight, reflection, or overall exposure difference
          - a registration seam or edge halo along the object's own boundary
            (typically a long thin strip)
          - blur, noise, or a rendering artifact with no physical cause
          - a normal design feature: a hole, rim, edge, seam or printed mark that is
            SUPPOSED to be there and is present in panel 1 too

Reply with exactly this JSON and nothing else:
{{"changed": true or false,
  "what_changed": "<what differs between panel 1 and panel 2, or \"nothing\">",
  "is_defect": true or false,
  "defect_type": "<short noun phrase, or \"none\">",
  "confidence": <0.0 to 1.0>,
  "reason": "<one sentence, max 25 words>",
  "artifact_kind": "<one of: texture, shadow, seam, blur, design_feature, none>"}}"""

# Fallback when registration produced no usable before-panel. Same task, weaker evidence --
# and it says so, so the model is not misled into thinking it has a reference it does not have.
Q_VLM2_2 = """Panel 1 (LEFT) is a crop from a photograph of a {cat} ({kind_h}).
Panel 2 (RIGHT) is the same crop with a candidate region outlined in red.

No before-image is available for this crop, so judge the outlined region on its own.
The region was proposed by an automatic difference detector, so check whether it is instead
normal texture, a shadow, a registration seam, or a design feature that is supposed to be
there -- but call it a defect if that is what it is.
{multi}
Reply with exactly this JSON and nothing else:
{{"changed": true or false,
  "what_changed": "<what looks anomalous, or \"nothing\">",
  "is_defect": true or false,
  "defect_type": "<short noun phrase, or \"none\">",
  "confidence": <0.0 to 1.0>,
  "reason": "<one sentence, max 25 words>",
  "artifact_kind": "<one of: texture, shadow, seam, blur, design_feature, none>"}}"""

MULTI_CLAUSE = """
NOTE: exactly ONE defect was introduced into this image, but {n} separate regions were
detected in it. This is region {k} of {n}, ranked {k} by area. At most one of the {n} can be
the introduced defect -- the others are extraction artifacts. A long thin strip is very
likely a registration seam rather than a defect. Judge THIS region on its own merits and be
correspondingly stricter.
"""

KIND_HUMAN = {"foreign_object": "defects here are usually foreign objects or contamination",
              "surface":        "defects here are usually surface damage, marks, or deformation",
              "unknown":        "defect appearance varies"}


def vlm_panels(entry, ctx=VLM_CTX):
    """[registered normal | anomaly | anomaly + outline] at identical coordinates.

    Returns (PIL image, has_reference). The crop is WIDER than the banked one (VLM_CTX vs
    CTX_EXPAND) so clean substrate stays in frame for the model to compare against.
    """
    d = RAW[entry["donor"]]; r = entry["_region"]
    anom, msk = d["anomaly"], d["mask"]
    H, W = anom.shape[:2]
    dh, dw = r["h"], r["w"]
    side = int(np.clip(round(max(dh, dw) * ctx), 48, min(H, W)))
    cy, cx = r["y"] + dh // 2, r["x"] + dw // 2
    ty = int(np.clip(cy - side // 2, 0, H - side))
    tx = int(np.clip(cx - side // 2, 0, W - side))

    a_crop = np.ascontiguousarray(anom[ty:ty + side, tx:tx + side])
    m_crop = (msk[ty:ty + side, tx:tx + side] > 127).astype(np.uint8)

    # The registered normal lives at diffmask's WORKING resolution, not the full frame, so the
    # box has to be rescaled before it can be cut from it.
    n_crop, ref = None, d.get("ref_aligned")
    if ref is not None and ref.size:
        fy, fx = ref.shape[0] / float(H), ref.shape[1] / float(W)
        ry, rx = int(round(ty * fy)), int(round(tx * fx))
        rh, rw = max(8, int(round(side * fy))), max(8, int(round(side * fx)))
        cand = ref[ry:ry + rh, rx:rx + rw]
        # Registration leaves black where the reference did not cover the frame. A mostly-black
        # before-panel is worse than none: it invites the model to call the difference a change.
        if cand.size and float((cand.reshape(-1, 3).max(1) > 8).mean()) > 0.90:
            n_crop = cv2.resize(cand, (side, side), interpolation=cv2.INTER_LINEAR)

    marked = a_crop.copy()
    cont, _ = cv2.findContours(m_crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(marked, cont, -1, (255, 0, 0), max(2, side // 160))

    panels = ([n_crop] if n_crop is not None else []) + [a_crop, marked]
    gap = np.full((side, 12, 3), 255, np.uint8)
    out = panels[0]
    for p in panels[1:]:
        out = np.concatenate([out, gap, p], axis=1)
    return Image.fromarray(out), (n_crop is not None)


def vlm2(entry):
    """Semantic validation of one crop. Never raises -- a bad reply is a reject with a reason."""
    img, has_ref = vlm_panels(entry)
    multi = ""
    if ONE_DEFECT_PER_IMAGE and entry.get("n_regions", 1) > 1:
        multi = MULTI_CLAUSE.format(n=entry["n_regions"], k=entry["region_rank"])
    tmpl = Q_VLM2_3 if has_ref else Q_VLM2_2
    q = tmpl.format(cat=entry["cat"].replace("_", " "), multi=multi,
                    kind_h=KIND_HUMAN.get(entry["kind"], KIND_HUMAN["unknown"]))
    raw, secs = vlm(img, SYS_VLM2, q, temperature=0.0)
    try:
        v = grab_json(raw)
        changed = bool(v.get("changed", True))
        raw_def = bool(v.get("is_defect", False))
        # A reply that says "nothing changed" and then ticks is_defect has contradicted itself.
        # Rejecting is the safe reading, but it is logged as its own class rather than folded
        # into "not a defect" -- if this fires often the prompt is confusing the model, which is
        # a prompt bug and not evidence about the crop.
        is_def = raw_def and changed
        contradiction = raw_def and not changed
        return dict(changed=changed, is_defect=is_def, contradiction=contradiction,
                    what_changed=str(v.get("what_changed", ""))[:120],
                    defect_type=str(v.get("defect_type", "none"))[:60],
                    confidence=float(v.get("confidence", 0.0)),
                    reason=str(v.get("reason", ""))[:200],
                    artifact_kind=str(v.get("artifact_kind", "none"))[:20],
                    has_ref=has_ref, raw=raw, secs=secs, parse_ok=True)
    except Exception as e:
        # An unparseable reply is a reject, not a crash, and it is recorded as its own class so
        # section 9 can separate "the model said no" from "the model said something unusable".
        return dict(changed=False, is_defect=False, contradiction=False, what_changed="",
                    defect_type="none", confidence=0.0,
                    reason=f"unparseable reply: {type(e).__name__}", artifact_kind="none",
                    has_ref=has_ref, raw=raw, secs=secs, parse_ok=False)


# The first call is timed like the rest but flagged: it carries CUDA warmup no later call pays.
for i, e in enumerate(CROPS):
    e["vlm"] = vlm2(e)
    e["vlm"]["warmup"] = (i == 0)

_pass = [e for e in CROPS if e["vlm"]["is_defect"] and e["vlm"]["confidence"] >= VLM_MIN_CONF]
_bad  = [e for e in CROPS if not e["vlm"]["parse_ok"]]
_secs = [e["vlm"]["secs"] for e in CROPS]
print(f"VLM-2: {len(_pass)}/{len(CROPS)} accepted at confidence >= {VLM_MIN_CONF} "
      f"| {len(_bad)} unparseable | {sum(_secs):.0f}s total\n")

_n_ref = sum(1 for e in CROPS if e["vlm"]["has_ref"])
print(f"{_n_ref}/{len(CROPS)} crops judged WITH a before-panel; "
      f"{len(CROPS)-_n_ref} fell back to the 2-panel prompt\n")
print(f"{'key':30s} {'ref':>4s} {'chg':>4s} {'acc':>4s} {'conf':>5s} {'secs':>6s} "
      f"{'artifact':14s} what changed")
for e in CROPS:
    v = e["vlm"]
    print(f"{e['key']:30s} {'3p' if v['has_ref'] else '2p':>4s} "
          f"{'yes' if v['changed'] else 'NO':>4s} "
          f"{'YES' if v['is_defect'] else 'no':>4s} {v['confidence']:5.2f} "
          f"{v['secs']:6.1f}{'*' if v['warmup'] else ' '} "
          f"{v['artifact_kind'][:14]:14s} {(v['what_changed'] or v['reason'])[:52]}")

# "nothing changed" is the answer the old single-image prompt could never give. If it never
# fires, the before-panel is not doing its job and the gate is still guessing from texture.
_nochg = [e for e in CROPS if not e["vlm"]["changed"]]
print(f"\n{len(_nochg)} crop(s) reported NO CHANGE between the before and after panels"
      + ("  <- these are the false positives the 3-panel prompt exists to catch"
         if _nochg else "  <- suspicious: check the before-panel is really being shown"))
for e in _nochg:
    print(f"    {e['key']:30s} {e['vlm']['reason'][:70]}")
if _secs:
    print(f"\n* first call, includes CUDA warmup")
    print(f"per crop: mean {np.mean(_secs):.1f}s | median {np.median(_secs):.1f}s "
          f"| min {min(_secs):.1f}s | max {max(_secs):.1f}s")

# Is the confidence number worth thresholding on? If it is flat, it is not, and
# USE_VLM_CONFIDENCE should stay off. Printed rather than assumed.
_cf = [e["vlm"]["confidence"] for e in CROPS]
if _cf:
    import numpy as _np
    _h, _ = _np.histogram(_cf, bins=[0, .2, .4, .6, .8, 1.01])
    print("\nconfidence distribution  "
          + "  ".join(f"{lo:.1f}-{hi:.1f}:{n}" for (lo, hi), n in
                      zip([(0,.2),(.2,.4),(.4,.6),(.6,.8),(.8,1.0)], _h)))
    print(f"  spread {min(_cf):.2f}-{max(_cf):.2f}"
          + ("   <- flat: do NOT gate on it (USE_VLM_CONFIDENCE stays False)"
             if max(_cf) - min(_cf) < 0.25 else "   <- has spread; gating on it is defensible"))

# What the accepted crops were CALLED. This is the bank's second index axis, not decoration --
# section 10 writes entries under <category>/<defect_type>/, per paper 3.3.
from collections import Counter
_types = Counter(e["vlm"]["defect_type"].strip().lower() for e in _pass)
print(f"\ndefect types named by VLM-2 across {len(_pass)} accepted crops:")
for t, n in _types.most_common():
    print(f"  {n:3d}  {t}")
if len(_types) == 1 and len(_pass) > 3:
    print("  -- one label for every crop. VLM-2 is acting as a pure yes/no gate here; the")
    print("     defect_type axis is carrying no information and the bank index is flat.")

## 8. Contrast gate

A second, independent check. It measures the CIELAB distance from each masked pixel to the crop's
own background, and requires a minimum fraction of them to exceed `DEFECT_T`.

This catches a mask that landed on plain background. Such an entry would produce an image whose
ground truth marks a region where nothing changed, so a detector would be scored on finding
something that is not there.

Both gate results are recorded separately, so either can be disabled without re-running the other.
The verdict overlay below colours every crop by the outcome: green is banked, red is discarded,
with the gate that rejected it.

In [ ]:
for e in CROPS:
    # Verdict alone unless USE_VLM_CONFIDENCE is switched on -- see the note in section 1.
    e["gate_vlm"]      = bool(e["vlm"]["is_defect"]) and (
        e["vlm"]["confidence"] >= VLM_MIN_CONF if USE_VLM_CONFIDENCE else True)
    e["gate_contrast"] = bool(e["cfrac"] >= MIN_ENTRY_CFRAC)
    e["gate_primary"]  = True                      # set below when ONE_DEFECT_PER_IMAGE is on
    e["accepted"]      = e["gate_vlm"] and e["gate_contrast"]

# Third gate: one defect per generated image. Among the crops from a donor that already passed
# both other gates, keep the one VLM-2 was most confident about and drop the rest. Ties break on
# area, because a seam artifact is usually the smaller of the two.
if ONE_DEFECT_PER_IMAGE:
    _cand = {}
    for e in CROPS:
        if e["accepted"]:
            _cand.setdefault(e["donor"], []).append(e)
    for d, rs in _cand.items():
        if len(rs) <= 1:
            continue
        rs.sort(key=lambda x: (-x["vlm"]["confidence"], -x["area_px"]))
        for e in rs[1:]:
            e["gate_primary"] = False
            e["accepted"] = False

BANK     = {c: [] for c in CATS}
REJECTED = []
for e in CROPS:
    (BANK[e["cat"]].append(e) if (e["accepted"] and e["cat"] in BANK) else REJECTED.append(e))

# Which gate did the work? If one gate never rejects anything the other one did not, it is not
# earning its place in the paper and 3.3 should say so.
_only_vlm  = [e for e in CROPS if not e["gate_vlm"] and e["gate_contrast"]]
_only_con  = [e for e in CROPS if e["gate_vlm"] and not e["gate_contrast"]]
_both_rej  = [e for e in CROPS if not e["gate_vlm"] and not e["gate_contrast"]]
_not_prim  = [e for e in CROPS if not e["gate_primary"]]

_n_bank = sum(len(v) for v in BANK.values())
print(f"accepted                    {_n_bank:3d}   from {len(ids)} anomaly images")
print(f"rejected by VLM-2 only      {len(_only_vlm):3d}   (contrast would have passed them)")
print(f"rejected by contrast only   {len(_only_con):3d}   (VLM-2 would have passed them)")
print(f"rejected by both            {len(_both_rej):3d}")
if ONE_DEFECT_PER_IMAGE:
    print(f"dropped as non-primary      {len(_not_prim):3d}   "
          f"(passed both gates, but their donor already had a better crop)")
    for e in _not_prim:
        best = max((x for x in CROPS if x["donor"] == e["donor"] and x["accepted"]),
                   key=lambda x: x["vlm"]["confidence"], default=None)
        print(f"    {e['key']:30s} conf {e['vlm']['confidence']:.2f} {e['area_px']:>7d}px"
              + (f"  <- kept {best['key'].split('_',1)[1]} instead" if best else ""))
    if _n_bank > len(ids):
        print("    NOTE: more crops than anomaly images -- ONE_DEFECT_PER_IMAGE did not bind.")
print()
for c in CATS:
    n = len(BANK[c])
    print(f"  {c:14s} {n:3d} entr{'y' if n == 1 else 'ies'}"
          f"{'   <- EMPTY, category unusable in Stages 4-5' if n == 0 else ''}")

In [ ]:
# --- THE VERDICT OVERLAY. Every crop, contoured, coloured by what the gates decided. --
# Green = banked, red = discarded. This is the figure to read before trusting the bank: the
# green panels are what Stages 4-5 will composite, and anything green that is not a defect
# becomes a wrong label in every downstream metric.
_ncol = 6
_nrow = int(math.ceil(len(CROPS) / _ncol))
fig, ax = plt.subplots(_nrow, _ncol, figsize=(2.7 * _ncol, 3.1 * _nrow), squeeze=False)
for k, e in enumerate(CROPS):
    crop = np.ascontiguousarray(e["rgb"].copy())
    cnts, _ = cv2.findContours((e["alpha"] > 0.5).astype(np.uint8),
                               cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    col = (0, 255, 0) if e["accepted"] else (255, 0, 0)
    cv2.drawContours(crop, cnts, -1, col, 2)
    a = ax[divmod(k, _ncol)]
    a.imshow(crop)
    why = "KEEP" if e["accepted"] else (
        ("vlm" if not e["gate_vlm"] else "")
        + ("+contrast" if not e["gate_contrast"] else "")
        + ("+non-primary" if not e["gate_primary"] else ""))
    a.set_title(f"{e['key'][-18:]}\n{why}  conf {e['vlm']['confidence']:.2f}  cf {e['cfrac']:.2f}",
                fontsize=7, color="green" if e["accepted"] else "red")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout(); plt.show()
print("Green = goes into defect_bank/. Red = discarded, with the gate that rejected it.")
print("Anything green that is not a defect belongs in the red set -- lower VLM_MIN_CONF or")
print("raise MIN_ENTRY_CFRAC. Anything red that IS a defect is a gate that is too strict.")

## 9. Agreement with manual labels

Optional. Compares the model's rejections against a list of regions you have rejected by hand.

Populate `HUMAN_DROP` with the keys you rejected, in the `<category>-<id>_<x>_<y>` format printed
in section 5. The cell prints a confusion table and, once enough labels exist, agreement, recall
on known artifacts and the false reject rate.

Below the minimum label count it prints counts only and refuses to quote a rate, because a single
label swings a recall by 100 points. The disagreement gallery is the place to start labelling.

In [ ]:
# Keys the human rejected, translated from build_flash_replay_nb.py's DROP_REGIONS.
# Its format is (f"{cat}-{donor}", x, y); this notebook's key is f"{cat}_{donor}_{x}_{y}".
#
# THE WHOLE LIST IS ONE ENTRY. That is the honest state of the human label set today, and it
# is not enough to support a paper claim -- one artifact cannot establish a recall. Growing
# this set is the prerequisite for section 3.3, not an optional extra.
HUMAN_DROP = {
    "fruit_jelly_064_789_916",      # cup base, not a defect -- documented in _command.txt
}
HUMAN_LABELLED = set()      # every key the human actually looked at; defaults to all crops
MIN_LABELS_FOR_RATE = 8     # below this, print counts but refuse to quote a rate


def agreement_report(crops, human_drop, human_seen):
    seen = human_seen or {e["key"] for e in crops}
    scored = [e for e in crops if e["key"] in seen]
    covered = human_drop & {e["key"] for e in scored}
    if not scored or not covered:
        print("No crop in this run matches a key in HUMAN_DROP -- cannot conclude.")
        print(f"HUMAN_DROP holds {len(human_drop)} key(s); none of them were recovered here.")
        print("Either this run used different pairs, or the region coordinates shifted.")
        return None

    tp = [e for e in scored if e["key"] in human_drop and not e["gate_vlm"]]   # both reject
    fn = [e for e in scored if e["key"] in human_drop and e["gate_vlm"]]       # VLM accepted an artifact
    fp = [e for e in scored if e["key"] not in human_drop and not e["gate_vlm"]]
    tn = [e for e in scored if e["key"] not in human_drop and e["gate_vlm"]]

    n, n_pos = len(scored), len(tp) + len(fn)
    print(f"scored {n} crops against {n_pos} human rejection(s)\n")
    print(f"{'':22s}{'VLM-2 reject':>14s}{'VLM-2 accept':>14s}")
    print(f"{'human reject':22s}{len(tp):>14d}{len(fn):>14d}")
    print(f"{'human keep':22s}{len(fp):>14d}{len(tn):>14d}")
    print()
    if n_pos < MIN_LABELS_FOR_RATE:
        print(f"NOT ENOUGH LABELS. {n_pos} human rejection(s) covered; "
              f"{MIN_LABELS_FOR_RATE} is the floor for quoting a rate.")
        print("The counts above are real. A recall computed from them is not, and must not")
        print("go in the paper -- one artifact caught or missed swings it by 100 points.")
        print("\nTo make section 3.3 measurable: label the regions this run recovered, add the")
        print("rejected keys to HUMAN_DROP, and put every key you looked at in HUMAN_LABELLED.")
    else:
        print(f"agreement                  {(len(tp) + len(tn)) / n:.1%}")
        print(f"recall on known artifacts  {len(tp) / n_pos:.1%}"
              f"   <- the number that matters")
        if (len(fp) + len(tn)):
            print(f"false-reject rate          {len(fp) / (len(fp) + len(tn)):.1%}"
                  f"   (costs bank size, not correctness)")
    return dict(tp=tp, fn=fn, fp=fp, tn=tn, n=n, n_pos=n_pos,
                conclusive=n_pos >= MIN_LABELS_FOR_RATE)


AGREE = agreement_report(CROPS, HUMAN_DROP, HUMAN_LABELLED)

In [ ]:
# --- Where they disagree. This gallery is the deliverable, whichever way the numbers fall. --
if AGREE:
    dis = AGREE["fn"] + AGREE["fp"]
    if not dis:
        print("no disagreements")
    else:
        fig, ax = plt.subplots(1, len(dis), figsize=(2.1 * len(dis), 2.6), squeeze=False)
        for j, e in enumerate(dis):
            marked = np.array(vlm_panels(e)[0])
            ax[0, j].imshow(marked); ax[0, j].axis("off")
            side = "VLM kept, human dropped" if e in AGREE["fn"] else "VLM dropped, human kept"
            ax[0, j].set_title(f"{e['key'][-14:]}\n{side}", fontsize=7)
        plt.tight_layout(); plt.show()
        for e in dis:
            print(f"{e['key']:38s} conf {e['vlm']['confidence']:.2f}  "
                  f"{e['vlm']['artifact_kind']:8s} {e['vlm']['reason']}")

## 10. Write the output

Writes the two folders described at the top of this notebook, plus a rejection log beside them
recording why each discarded crop was dropped.

The bank holds accepted crops only. `manifest.csv` carries the geometry Module 3 needs, including
`defect_frac`, which cannot be recovered from the PNG and without which defects replay at the
wrong size.

Any pair whose mask came back empty is removed from the second folder at this point.

In [ ]:
import csv

# --- Folder 1: the defect bank. Accepted cropped patches, nothing else. -------------
if os.path.isdir(BANK_DIR):
    shutil.rmtree(BANK_DIR)          # a stale entry from a previous run is worse than no bank
os.makedirs(BANK_DIR, exist_ok=True)

MANIFEST_COLS = ["key", "cat", "donor", "kind", "defect_type", "side", "defect_long_px",
                 "defect_frac", "area_px", "area_frac", "substrate_frac", "contrast", "cfrac",
                 "vlm_conf", "vlm_changed", "vlm_contradiction", "vlm_what_changed", "vlm_has_ref",
                 "vlm_reason", "gate_vlm", "gate_contrast", "gate_primary",
                 "n_regions", "region_rank",
                 "diffmask_secs", "vlm_secs", "flags"]

def row_of(e):
    v = e["vlm"]
    return dict(key=e["key"], cat=e["cat"], donor=e["donor"], kind=e["kind"],
                defect_type=v["defect_type"], side=e["side"],
                defect_long_px=e["defect_long_px"], defect_frac=round(e["defect_frac"], 5),
                area_px=e["area_px"], area_frac=round(e["area_frac"], 6),
                substrate_frac=round(e["substrate_frac"], 4),
                contrast=round(e["contrast"], 2), cfrac=round(e["cfrac"], 4),
                vlm_conf=round(v["confidence"], 3), vlm_changed=v["changed"],
                vlm_contradiction=v.get("contradiction", False),
                vlm_what_changed=v["what_changed"], vlm_has_ref=v["has_ref"],
                vlm_reason=v["reason"],
                gate_vlm=e["gate_vlm"], gate_contrast=e["gate_contrast"],
                gate_primary=e["gate_primary"], n_regions=e.get("n_regions", 1),
                region_rank=e.get("region_rank", 1),
                diffmask_secs=round(e["diffmask_secs"], 2), vlm_secs=round(v["secs"], 2),
                flags=e["flags"])

n_written = 0
for cat in CATS:
    for e in BANK[cat]:
        d = os.path.join(BANK_DIR, cat); os.makedirs(d, exist_ok=True)
        cv2.imwrite(os.path.join(d, f"{e['key']}.png"),
                    cv2.cvtColor(e["rgb"], cv2.COLOR_RGB2BGR))
        # The alpha is NOT optional even though only "the patch" was asked for: Stages 4-5 call
        # place_entry(entry["rgb"], entry["alpha"], ...) and cannot composite without it. It is
        # a sibling file rather than an RGBA channel so the crop keeps its substrate -- an RGBA
        # cut-out leaves the harmoniser nothing to measure.
        cv2.imwrite(os.path.join(d, f"{e['key']}_alpha.png"),
                    np.clip(e["alpha"] * 255, 0, 255).astype(np.uint8))
        n_written += 1

with open(os.path.join(BANK_DIR, "manifest.csv"), "w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=MANIFEST_COLS)
    w.writeheader()
    w.writerows([row_of(e) for c in CATS for e in BANK[c]])

# Rejection log. It sits at the ROOT, outside both deliverable folders, so defect_bank/ still
# holds only patches -- but "why did this category only produce two" has to be answerable after
# the run, not just from console output that scrolls away.
def _why(e):
    if not e["gate_vlm"]:
        if e["vlm"].get("contradiction"):
            return "vlm_contradiction"
        if not e["vlm"]["parse_ok"]:
            return "vlm_unparseable"
        if not e["vlm"]["changed"]:
            return "vlm_no_change"
        if USE_VLM_CONFIDENCE and e["vlm"]["confidence"] < VLM_MIN_CONF:
            return "vlm_low_confidence"
        return "vlm_not_defect"
    if not e["gate_contrast"]:
        return "low_contrast"
    if not e["gate_primary"]:
        return "not_primary_region"
    return "?"

_rej = [e for e in CROPS if not e["accepted"]]
with open(os.path.join(OUT_ROOT, "stage23_rejected.csv"), "w", newline="",
          encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=["reason"] + MANIFEST_COLS)
    w.writeheader()
    for e in _rej:
        w.writerow(dict(reason=_why(e), **row_of(e)))

print(f"defect_bank/            {n_written} accepted patches")
if _rej:
    from collections import Counter as _C
    print(f"stage23_rejected.csv    {len(_rej)} discarded: "
          + ", ".join(f"{n} {r}" for r, n in _C(_why(e) for e in _rej).most_common()))
for c in CATS:
    if BANK[c]:
        print(f"  {c:14s} {len(BANK[c]):3d}")

In [ ]:
# --- Folder 2: (recovered mask, generated anomaly) pairs, category-wise -------------
# Written by section 4 as diffmask produced them. Counted here, and pairs whose mask came back
# empty are removed: a mask with no defect in it is not a pair, it is a failure to recover one.
n_pairs = 0
for cat in sorted(os.listdir(PAIRS_DIR)):
    d = os.path.join(PAIRS_DIR, cat)
    if not os.path.isdir(d):
        continue
    for mp in sorted(glob.glob(os.path.join(d, "*_mask.png"))):
        m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
        if m is None or (m > 127).sum() == 0:
            os.remove(mp)
            ap = mp.replace("_mask.png", "_anomaly.png")
            if os.path.exists(ap):
                os.remove(ap)
        else:
            n_pairs += 1

print(f"Defect Masks and Anomalies/  {n_pairs} pairs")
for cat in sorted(os.listdir(PAIRS_DIR)):
    d = os.path.join(PAIRS_DIR, cat)
    if os.path.isdir(d):
        print(f"  {cat:14s} {len(glob.glob(os.path.join(d, '*_mask.png'))):3d}")

# Provenance, one file, beside the two folders rather than inside either.
json.dump(dict(seed=SEED, pairs_root=PAIRS_ROOT, model=MODEL_PATH,
               flags=" ".join(FLAGS), category_agnostic=True,
               ctx_expand=CTX_EXPAND, min_region_px=MIN_REGION_PX, alpha_soften=ALPHA_SOFTEN,
               defect_t=DEFECT_T, min_entry_cfrac=MIN_ENTRY_CFRAC, vlm_min_conf=VLM_MIN_CONF,
               work_size=WORK_SIZE, n_pairs=len(PAIRS), n_crops=len(CROPS),
               n_banked=n_written, n_mask_pairs=n_pairs,
               load_seconds=round(LOAD_SECONDS, 1),
               diffmask_seconds=round(sum(r["secs"] for r in RESULTS), 1),
               vlm_seconds=round(sum(e["vlm"]["secs"] for e in CROPS), 1)),
          open(os.path.join(OUT_ROOT, "stage23_run_config.json"), "w"), indent=2)

In [ ]:
from IPython.display import FileLink, display

for name, src in (("defect_bank", BANK_DIR),
                  ("Defect Masks and Anomalies", PAIRS_DIR)):
    zp = os.path.join(OUT_ROOT, f"{name}.zip")
    if os.path.exists(zp):
        os.remove(zp)
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for root, _, files in os.walk(src):
            for f in files:
                full = os.path.join(root, f)
                z.write(full, os.path.relpath(full, OUT_ROOT))
    print(f"{zp}  ({os.path.getsize(zp)/1e6:.1f} MB)")
    display(FileLink(zp))

## 11. Run summary

The per-category funnel: pairs, recovered masks, candidate crops, crops passing validation, and
crops banked, with the reason for each loss.

A category reaching zero cannot be used by Module 3 and will drop out of any evaluation. If that
happens, look at the section 5 overlays for that category before changing any threshold.

In [ ]:
print(f"{'category':14s} {'split':9s} {'pairs':>6s} {'masks':>6s} {'crops':>6s} {'vlm':>5s}"
      f" {'banked':>7s} {'empty':>6s} {'fail':>5s}   why the crops were lost")
tot = dict(pairs=0, masks=0, crops=0, vlm=0, banked=0, empty=0, fail=0)
for c in sorted({r["cat"] for r in RESULTS}):
    rows  = [r for r in RESULTS if r["cat"] == c]
    crops = [e for e in CROPS if e["cat"] == c]
    row = dict(pairs=len(rows),
               masks=len([r for r in rows if r["err"] is None and r["mask_px"] > 0]),
               crops=len(crops),
               vlm=len([e for e in crops if e["gate_vlm"]]),
               banked=len(BANK.get(c, [])),
               empty=len([r for r in rows if r["err"] is None and r["mask_px"] == 0]),
               fail=len([r for r in rows if r["err"] is not None]))
    for k in tot:
        tot[k] += row[k]
    split = "dev" if c in DEV_CATS else ("held-out" if c in HELDOUT_CATS else "?")
    lost = [e for e in crops if not e["accepted"]]
    from collections import Counter as _C2
    why = "  ".join(f"{n}x{r}" for r, n in _C2(_why(e) for e in lost).most_common()) or "-"
    print(f"{c:14s} {split:9s} {row['pairs']:6d} {row['masks']:6d} {row['crops']:6d}"
          f" {row['vlm']:5d} {row['banked']:7d} {row['empty']:6d} {row['fail']:5d}   {why}"
          f"{'   <- EMPTY' if row['banked'] == 0 else ''}")
print(f"{'TOTAL':14s} {'':9s} {tot['pairs']:6d} {tot['masks']:6d} {tot['crops']:6d}"
      f" {tot['vlm']:5d} {tot['banked']:7d} {tot['empty']:6d} {tot['fail']:5d}")

# The split that matters. Held-out yield is the number supplementary 3.A's claim rests on:
# the same flags, never retuned, applied to categories they were not fitted on.
for name, group in (("dev", DEV_CATS), ("held-out", HELDOUT_CATS)):
    rows  = [r for r in RESULTS if r["cat"] in group]
    crops = [e for e in CROPS if e["cat"] in group]
    bank  = sum(len(BANK.get(c, [])) for c in group)
    if rows:
        print(f"{name:>10s}: {len(rows):3d} pairs -> {bank:3d} banked "
              f"({bank / max(len(rows), 1):.2f} entries/pair, "
              f"{len(crops)} crops)")

### Timing

Per-pair recovery time and per-crop validation time, reported separately.

Model load is excluded because it is paid once per session. The two stages scale differently:
recovery runs once per pair, validation once per recovered crop, so a pair yielding three regions
costs one recovery and three validations.

The figure to quote is the amortised cost per banked entry, since rejected crops cost time and
yield nothing.

In [ ]:
print(f"model load (one-off, excluded below) : {LOAD_SECONDS:7.1f}s\n")

print(f"{'category':<14s}{'pairs':>7s}{'dm total':>10s}{'dm mean':>9s}"
      f"{'crops':>7s}{'vlm total':>11s}{'vlm mean':>10s}")
print("-" * 68)
_t = dict(pairs=0, dm=0.0, crops=0, vlm=0.0)
for c in sorted({r["cat"] for r in RESULTS}):
    rows  = [r for r in RESULTS if r["cat"] == c]
    crops = [e for e in CROPS if e["cat"] == c]
    dm  = sum(r["secs"] for r in rows)
    vl  = sum(e["vlm"]["secs"] for e in crops)
    _t["pairs"] += len(rows); _t["dm"] += dm
    _t["crops"] += len(crops); _t["vlm"] += vl
    print(f"{c:<14s}{len(rows):>7d}{dm:>10.1f}{dm / max(len(rows), 1):>9.1f}"
          f"{len(crops):>7d}{vl:>11.1f}{vl / max(len(crops), 1):>10.1f}")
print("-" * 68)
print(f"{'ALL':<14s}{_t['pairs']:>7d}{_t['dm']:>10.1f}{_t['dm'] / max(_t['pairs'], 1):>9.1f}"
      f"{_t['crops']:>7d}{_t['vlm']:>11.1f}{_t['vlm'] / max(_t['crops'], 1):>10.1f}")

# Warm figure: the first VLM call pays CUDA warmup that nothing after it does.
_warm = [e["vlm"]["secs"] for e in CROPS if not e["vlm"]["warmup"]]
_cold = [e["vlm"]["secs"] for e in CROPS if e["vlm"]["warmup"]]
print(f"\nStage 2  DiffMask  {_t['dm']:8.1f}s  over {_t['pairs']:3d} pairs"
      f"   {_t['dm'] / max(_t['pairs'], 1):6.2f}s/pair")
print(f"Stage 3  VLM-2     {_t['vlm']:8.1f}s  over {_t['crops']:3d} crops"
      f"   {_t['vlm'] / max(_t['crops'], 1):6.2f}s/crop")
if _warm:
    _cold_note = f"   (cold first call {_cold[0]:.1f}s)" if _cold else ""
    print(f"         VLM-2 warm  {len(_warm):8d} crops"
          f"   {np.mean(_warm):6.2f}s/crop  median {np.median(_warm):.2f}s{_cold_note}")
print(f"{'':9s}{'-' * 44}")
print(f"Stages 2-3 total   {_t['dm'] + _t['vlm']:8.1f}s"
      f"   (+ {LOAD_SECONDS:.0f}s one-off model load)")

_banked = sum(len(v) for v in BANK.values())
if _banked:
    print(f"\nAmortised cost per BANKED entry : "
          f"{(_t['dm'] + _t['vlm']) / _banked:.1f}s")
    print("Per banked entry, not per crop -- rejected crops cost time and yield nothing, so")
    print("they belong in the numerator. This is the number the paper should quote.")

## Before using this bank

1. Every banked crop shows visible background in the section 5 overlay. A crop that is almost
   entirely defect will place badly in Module 3.
2. No category is empty. An empty category disappears silently from Module 3.
3. `defect_type` is not the same label repeated. Section 7 warns if it is.
4. If you changed the DiffMask flags, re-check every category, not just the one you were fixing.